# MoLFormer-XL — DIMER E2E molecular-property classification tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/molformer-chemistry-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/molformer-chemistry-pipeline/blob/main/tutorials/molformer_chemistry_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-ibm--research%2FMoLFormer--XL--both--10pct-ffcc4d?style=flat)](https://huggingface.co/ibm-research/MoLFormer-XL-both-10pct) [![Upstream](https://img.shields.io/badge/Upstream-IBM%2Fmolformer-181717?style=flat&logo=github&logoColor=white)](https://github.com/IBM/molformer) [![arXiv](https://img.shields.io/badge/arXiv-2106.09553-b31b1b.svg)](https://arxiv.org/abs/2106.09553)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.2 — **standalone** (§4)  
**Capability:** SMILES embeddings and bounded molecular-property classification fine-tuning

**This notebook is standalone.** It carries the repository's package (3 modules under `src/molformer_chemistry_pipeline/`, at revision `6cde8bd0b92f`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `361063d0ad524ef77cf39b08469f6be770dc550f` (~187 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2.2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned MoLFormer-XL snapshot (7 files, ~187 MB, including the two Python files the loader executes), loads the tokenizer without remote code and the model with it, generates a deterministic 64-molecule constitutional-isomer dataset in code (no download), validates the SMILES and the dataset contract, splits it into stratified train/validation/test sets, shows how SMILES are tokenized, computes mean-pooled molecule embeddings, measures a majority-class and a molecular-formula baseline on the test split, runs a bounded AdamW fine-tuning of the property-classification head and the last two encoder layers, evaluates accuracy, macro-F1 and AUROC on the held-out test split, classifies six freshly generated molecules, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify prediction parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes well under a minute of model time.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` and `BYOD_PATH` (a file already in the runtime; on Colab an empty path opens an upload dialog) in Section 4 and re-run from that cell to supply your own labelled molecules as a CSV (`id,smiles,label`), a JSON array or a JSONL file. They pass through the same SMILES validation, stratified split, baselines, adaptation, held-out evaluation, inference, artifact export and reload-parity cells as the synthetic sample. The expected schema, the accepted SMILES characters and the token ceiling are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

MoLFormer-XL is a chemical language model: it reads molecules written as **SMILES strings** and was pretrained with a masked-token objective on a very large corpus of them (the upstream card describes 1.1 billion molecules from PubChem and ZINC; this checkpoint is the variant trained on a 10 % sample of both). Its notable architectural choice is **linear attention with random feature maps**, which is why it scales to that corpus — and why it behaves differently from an ordinary transformer in one respect that matters for reproducibility, explained in Section 3.

This tutorial adapts it to a molecular-property task. The tutorial dataset is synthetic but chemically literal: every molecule is paired with a **constitutional isomer** of itself — the same molecular formula, the same atoms, a different skeleton. One member is an unbranched chain (`CCCCCCCO`); the other carries one of those carbons as a methyl branch (`CCCC(C)CCO`). A classifier that only knows the molecular formula therefore cannot do better than chance, by construction, which is what makes the fine-tuned model's result worth reading.

**Learning objectives:** install the pinned runtime; inspect the carried pipeline, dataset and metrics modules; stage and digest-verify an immutable snapshot **including the Python the loader executes**; state what `trust_remote_code=True` executes and what the digest check does and does not protect against, and check that the `deterministic_eval` override makes repeated embeddings identical; validate SMILES syntactically and split a labelled dataset without leakage; read how SMILES become tokens; extract mean-pooled molecule embeddings; measure majority-class and molecular-formula baselines; run a bounded fine-tuning with explicit hyperparameters; evaluate accuracy, macro-F1 and AUROC on an independent test split; classify new molecules; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** regression on continuous properties, the published MoleculeNet benchmarks, molecule generation or optimisation, 3D conformers or descriptors, chemical validity checking (no RDKit is installed), and the other MoLFormer checkpoints. The repository exposes none of these.

**Who this notebook is for.** A learner who knows basic Python, has used Colab or Jupyter and has met the idea of a classifier, and wants to see how a chemical language model reads molecules written as SMILES, how to test whether it has learned structure rather than composition, and how to fine-tune it on a small labelled set and export the result. The audience is chemistry and data-science students and practitioners preparing their own molecular-property data; no prior experience with MoLFormer, transformers or fine-tuning is assumed — each term is explained where it first matters and again in the **Glossary**. CPU is enough: the default fine-tuning takes seconds.

**Input → Model → Output.**

| | Embeddings | Bounded fine-tuning and classification |
|---|---|---|
| Input | SMILES strings (at most 200 tokens each) | labelled records `{id, smiles, label}`: 64 generated constitutional-isomer molecules (36 train, 12 validation, 16 test) or your own CSV/JSON/JSONL |
| Model | the MoLFormer-XL encoder (linear attention with random feature maps, loaded with deterministic evaluation) | the same encoder with a new classification head; the head and the last `TRAINABLE_LAYERS` encoder layers are trained with AdamW |
| Output | one 768-dimensional mean-pooled vector per molecule | an argmax label and softmax scores per molecule, test accuracy / macro-F1 / AUROC beside two baselines, and a safetensors adapter that reloads with identical scores |

**How to use this notebook.** Choose any runtime (CPU is enough), then **Runtime → Run all**. Run all completes in one pass: Section 1 installs nothing into the notebook's own Python, so no restart is needed (the recorded hosted run of the previous version needed one; this version removes it). Sections 1–3 are **infrastructure** — the isolated environment, the carried modules and the verified snapshot with its remote code — and their cells are collapsed; you may run them without studying them. The learning path starts in Section 4. Form fields (`# @param`) are the only values meant to be edited, and the defaults reproduce the recorded run. Before each principal result the notebook asks you to **Predict**; after it comes a collapsible **Check your reasoning** with a worked answer from the recorded runs (the Kaggle T4 run of 18 September 2026 and the local CPU pre-flight, which agree). **Troubleshooting**, a **Glossary** and a **Conclusion** template are at the end. Writing your predictions down is optional.

**Roadmap:** 1–3 infrastructure → 4 the isomer dataset, validation and split *(evaluation practice: a dataset built to defeat a shortcut)* → 5 how SMILES become tokens *(core concept)* → 6 embeddings *(core concept: representation, not prediction)* → 7 two baselines *(evaluation practice)* → 8 bounded fine-tuning *(core concept: what is trained)* → 9 held-out evaluation → 10 new molecules, export and fresh reload *(engineering)* → 11 result export → conclude.

## Prerequisites

- **Learner:** basic Python and Colab or Jupyter familiarity; no prior experience with MoLFormer, transformers or fine-tuning. SMILES, tokens, embeddings, baselines, accuracy / macro-F1 / AUROC, epochs and adapters are explained where they are first used and again in the Glossary.
- **Runtime:** a fresh supported **Linux x86_64** runtime (Google Colab, Kaggle or Linux Jupyter). Section 1 builds its own Python 3.12.12 environment from a hash-locked list of manylinux wheels, so the kernel's own Python version does not matter and nothing is installed into it; a Windows or macOS kernel is not supported. CPU is enough — the default fine-tuning is a couple of seconds — and CUDA is used automatically when present.
- **Knowledge:** how a molecule is written as SMILES, what a constitutional isomer is, and how accuracy, macro-F1 and AUROC differ.
- **Remote code:** the default path executes the checkpoint's own `configuration_molformer.py` and `modeling_molformer.py` after verifying their SHA-256 against the inline manifest. Section 3 explains why that is unavoidable for this checkpoint.
- **Runtime pin:** this repository pins `transformers==5.17.0`, not the fleet's 4.57.6, because the pinned upstream code calls an API that exists only in Transformers 5.5.0 and later.
- **Data contract:** records are `{id, smiles, label}`; SMILES are non-empty strings over the accepted character set, at most 200 tokens (`MAX_TOKENS`, the checkpoint's 202 position embeddings minus `<bos>`/`<eos>`), with unique ids and unique SMILES, 2..20 classes. The validation and test splits are each held to at least 8 records and 3 per class, so for two balanced classes at the default split fractions you need **at least 36 records (18 per class)**; more classes or imbalance raise that, and Section 4 refuses a dataset that is too small (naming the split and the real minimum) before any model runs. BYOD accepts CSV, JSON array or JSONL, read from `BYOD_PATH` (works on Colab, Kaggle and Jupyter) or, on Colab with the path left empty, from an upload dialog.
- **Validation is syntactic, not chemical:** this repository ships no cheminformatics toolkit, so it checks characters, bracket balance and ring-digit pairing — not valence or chemical plausibility. Use RDKit before trusting a SMILES set.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — an unpublished or third-party proprietary structure is exactly that. The default path uploads nothing.
- **External access:** the Hugging Face Hub only, to fetch the pinned `ibm-research/MoLFormer-XL-both-10pct` snapshot (~187 MB in total) at revision `361063d0ad52…`. No GitHub access and no credentials are required; nothing is installed from this repository. The isolated environment also needs PyPI (`pypi.org`, `files.pythonhosted.org`) for the pinned `uv` wheel and the 56 hash-locked packages, and the managed CPython 3.12.12 build (python-build-standalone) that `uv` downloads.

## 1. Install the pinned runtime (isolated environment)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

Hosted runtimes such as Colab and Kaggle import some packages, NumPy and often PyTorch among them, before the first cell runs, and Python cannot swap a module that is already loaded. Installing the pins into the notebook's own Python would therefore leave mixed versions or require a manual restart. Instead, the next cell builds a separate environment (`dimer_isolated_env_<lock digest>/`) and leaves the kernel's packages untouched: it downloads the pinned `uv` 0.12.15 wheel and refuses it unless its size and SHA-256 match, has `uv` install the managed CPython **3.12.12** (whatever Python the kernel itself runs), and installs the 56 packages of the carried hash-locked requirements (`tutorials/requirements-colab.lock.txt`, compiled from the pins in the cell) with `--require-hashes --only-binary :all:`, so every package, direct or transitive, is the exact file that was locked. No repository code is installed. The lock holds manylinux x86_64 wheels, so the notebook supports **Linux x86_64 runtimes only** (Google Colab, Kaggle or Linux Jupyter); on any other platform the cell stops with that message.

The same cell then starts one Python process in that environment and routes **every later code cell** to it. Printed output comes back to the notebook as usual, variables persist from cell to cell, and an error stops **Run all** as it would in the kernel. It is the only cell that runs in the kernel (it is marked `# dimer: kernel cell`), and it is safe to re-run: a complete environment built from this exact lock is reused rather than rebuilt, and a live worker is kept with every variable created so far, so the cells after it keep working. To start over, restart the session and choose **Run all**. `DIMER_NOTEBOOK_CI_PREINSTALLED=1` lets an executor that has already installed exactly these pins run every cell in its own kernel instead.

In [ ]:
# @title Infrastructure: build (or reuse) the isolated environment and route later cells to it
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import hashlib
import io
import os
import platform
import signal
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from multiprocessing.connection import Connection
from pathlib import Path

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==5.17.0',
    'huggingface-hub==1.32.0',
    'tokenizers==0.23.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
MANAGED_PYTHON = '3.12.12'
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
LOCK_NAME = 'requirements-colab.lock.txt'
LOCK_SHA256 = 'cbaf70c710fc48fbc7ffe6532d7dd7be222478bc3a411eca129134ee619d592f'
LOCKED_PACKAGES = 56
# The hash-locked requirements, compiled from the PINS above with `uv pip compile --generate-hashes` for manylinux x86_64.
LOCK_TEXT = r'''# This file was autogenerated by uv via the following command:
#    uv pip compile pyproject.toml --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt
annotated-doc==0.0.5 \
    --hash=sha256:117bac03a25ede5df5440e855b32d556049ca169ead221505badf432fed4b101 \
    --hash=sha256:c7e58ce09192557605d8bbd92836d7e1d520ac9580096042c0bfd197efacf1bb
    # via typer
anyio==4.15.1 \
    --hash=sha256:6152fdbbf9a77fdec97731721bebf7c4c44f7c29b424b0065826173efc7ed101 \
    --hash=sha256:9f28306018cbd6d329e64a36d58256edff76dd996fe423bc957326e578b82a94
    # via httpx
certifi==2026.7.22 \
    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \
    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55
    # via
    #   httpcore
    #   httpx
click==8.5.0 \
    --hash=sha256:255bc9599cf7748b4b1a446ccc735421bd08a2ae529a8b88597d3de5664ee360 \
    --hash=sha256:ba0d2089de75ea0310e2dde03160e6ca10009947fb95a182f9b54021bb272e34
    # via huggingface-hub
cuda-bindings==13.4.3 \
    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \
    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \
    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \
    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \
    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \
    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \
    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \
    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \
    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \
    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \
    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \
    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \
    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \
    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \
    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \
    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \
    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \
    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \
    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \
    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \
    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \
    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \
    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368
    # via torch
cuda-pathfinder==1.8.3 \
    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f
    # via cuda-bindings
cuda-toolkit==13.0.3.0 \
    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f
    # via torch
filelock==4.0.12 \
    --hash=sha256:5f17ee83ecee8a6f3e389c75822fb70a1c2f0506b99438a6dffa1d56793588c8 \
    --hash=sha256:cf42711a7ac791818b299fab0332a088c65aeeefa36290de98db92c434303b0c
    # via
    #   huggingface-hub
    #   torch
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
    # via
    #   huggingface-hub
    #   torch
h11==0.16.0 \
    --hash=sha256:4e35b956cf45792e4caa5885e69fba00bdbc6ffafbfa020300e549b208ee5ff1 \
    --hash=sha256:63cf8bbe7522de3bf65932fda1d9c2772064ffb3dae62d55932da54b31cb6c86
    # via httpcore
hf-xet==1.6.0 \
    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \
    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \
    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \
    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \
    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \
    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \
    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \
    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \
    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \
    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \
    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \
    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \
    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \
    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \
    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \
    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \
    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b
    # via huggingface-hub
httpcore==1.0.9 \
    --hash=sha256:2d400746a40668fc9dec9810239072b40b4484b640a8c38fd654a024c7a1bf55 \
    --hash=sha256:6e34463af53fd2ab5d807f399a9b45ea31c3dfa2276f15a2c3f00afff6e176e8
    # via httpx
httpx==0.28.1 \
    --hash=sha256:75e98c5f16b0f35b567856f597f06ff2270a374470a5c2392242528e3e3e42fc \
    --hash=sha256:d909fcccc110f8c7faf814ca82a9a4d816bc5a6dbfea25d6591d6985b8ba59ad
    # via huggingface-hub
huggingface-hub==1.32.0 \
    --hash=sha256:b0c7c80561969d9cdacdd55fce67ba9584cca0b9d4ea80957a3a5c1445fac5c8 \
    --hash=sha256:ed70a45498abe86039df7c2f4e5f7575de524be908d3840e8f828d5525eafd6a
    # via
    #   molformer-chemistry-pipeline (pyproject.toml)
    #   tokenizers
    #   transformers
idna==3.20 \
    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \
    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c
    # via
    #   anyio
    #   httpx
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
    # via torch
markdown-it-py==4.2.0 \
    --hash=sha256:04a21681d6fbb623de53f6f364d352309d4094dd4194040a10fd51833e418d49 \
    --hash=sha256:9f7ebbcd14fe59494226453aed97c1070d83f8d24b6fc3a3bcf9a38092641c4a
    # via rich
markupsafe==3.0.4 \
    --hash=sha256:007e1ffd9bf65bb6ee96df7b258fc632a4868dd5566037986c64781f35a36e98 \
    --hash=sha256:02fa4acbc6a3fc5c693c34d4dd8c1130b7fe99cc915181b0ddd6f72aeb296002 \
    --hash=sha256:03470d1a8268e692ecf79ecd565593e59d44219377a7ead61f1f1b94c1f7ff6b \
    --hash=sha256:04e7902ba80ee4bac1d50a549606527a1dcf0476cd81403db41099d3b60ec653 \
    --hash=sha256:051417f74bcaaefa316276e0ff723f541616ca51043d070da00249d9bddd3e3c \
    --hash=sha256:05295589e619b9bed252a86b532b8e27350abc372d18ba89b59375325e91ec1e \
    --hash=sha256:06de8ef6331f6e822c28d577dc8bf43fe398800477c49498f38fc38b67ff33fc \
    --hash=sha256:0764a13d34cae40db7bbf3a09b7e9b491bf4603e20b263a7a9d6b8e324975d0a \
    --hash=sha256:077293e425f28ec737dbcad442a71752e28f8ae27cde3d68acd1fb212091cd92 \
    --hash=sha256:0930db9bdc62d22944e10b066448bb65dc9abe9112880c7cab8da54db4284d5f \
    --hash=sha256:0cee7cb0f9a1b6892ea482237d9403b3d1b4603aee057d0ff01f0fac2d019a97 \
    --hash=sha256:0d9c47709875fdb321452056622e930c52afbc07a7d780762fbb8b4d91ce6fa4 \
    --hash=sha256:11935df9bf455ed0c04eb87bcd720f02b1fe5e02128a9430f23aed6f93336fc7 \
    --hash=sha256:12a606a492de952afcb43b59a14aaaaad120e708d3663dd0fdf2d738d427a691 \
    --hash=sha256:14bd2d845d62ab678eaf81da89d7b621b51756c72346745c1a594c09d49207a2 \
    --hash=sha256:15ba9e28640feef770374b116a6f019c21f52404aeabe516aa7f800587b98cfc \
    --hash=sha256:18a801868a884f216e784d7d14db2a4077143ce7610440aee2ce8f734e7cfcde \
    --hash=sha256:1c0df495a977d10460a94941799c72d5b5ab03d3858d949b55b5a66c8f371c99 \
    --hash=sha256:1caa2fa5a6184fb233153b35f654e6687bd555476f6170f29d8ee9be1a8b0af9 \
    --hash=sha256:1e1451fab512d1bcc3dc26988ec1edb0b82c2db909132872cd9356070a6b63df \
    --hash=sha256:1f1f9477e174582b0a1b583d60b66e1f2cf5d3fe12cee985e4aedf44766600e5 \
    --hash=sha256:2628d3a8cb648ecebb3c5d6b0a1052d400e4d8b7ac0fb786be8d285b50040d17 \
    --hash=sha256:26e9867520db70d37f7fb421a7f0d8adb40171011fb84ce869afa1a83370dfa8 \
    --hash=sha256:2a6ef68ae94aed8721934072b27a3b654ea2100b97e4ab864cf1489c90926fbc \
    --hash=sha256:2b2b1e18af909b448bb3cf9e3433366f7a8726271fc214e8b10e0f62a78c724b \
    --hash=sha256:2cb3dd71fc6be918ad4264346a8ed69485f9b7ed7bf35495d8e22807cd6b8bea \
    --hash=sha256:2d1b7d9308288661f56672b1b157d75fc536714d3638487bbea17b6318a78248 \
    --hash=sha256:2dad610540cb2e6272855c178f08ae9a1c7ac258a7fb71660553a5f104b42741 \
    --hash=sha256:2e5a7cd7fdd14fcb1ae5d7d8bf23d24fbd1daefd1fbca2580132e1ea75f098b5 \
    --hash=sha256:2e9ad7dd851bf45fab9f75cbff4cb493fee9979e8d8c7c9c3ee119022518edd6 \
    --hash=sha256:340cbb1957ba99929cbf19a75626d36ba1ae21d1730b287d1cf7f824a20c4fc7 \
    --hash=sha256:34bdde374c5932765d7dc685c4a1d191a3207852d67e8e0a9eb6ea85156181f1 \
    --hash=sha256:353bd63081912ab8cfa6a0c7d185934cdf8426f04c618bba6bc4b394f2069b67 \
    --hash=sha256:387d8cd30e69b3f0a72877b9ae717033396404e19095b17fe89753a981fda44f \
    --hash=sha256:3882fb412298575bae3b9c46868251f15cc69307359f87bb1b382e53d6e5a2c9 \
    --hash=sha256:38fc55594dab834470b6733dead2ee9e3f657fb0608c769dcafa0ba5ab52f45c \
    --hash=sha256:396ec4e65cc889f69786b3b89478b471cee5a3bcf468b9d9bb03e1a30fb291fc \
    --hash=sha256:39dbacefc411633db5b4378b066a9aca70a3d7e2922c9e578d825f844026eeba \
    --hash=sha256:3a93d9616ddecfb393727a0041a562cf0b15a244e20f2bd25efc7949be4c4f17 \
    --hash=sha256:3d23795802fc8bd72534836d64489bbf0f67c088959091bdb22e10735a5107bf \
    --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6 \
    --hash=sha256:436e3ffc6310d3c41878c601db29098102fe5d8a467c49da4a4125254e0980f2 \
    --hash=sha256:489505b03f692c3f376394e49194fa7a7f9e8558d6e293a7056a0032b0c38163 \
    --hash=sha256:4a540e2d3192792fc84eced57bef37851ccb2b41f73291bb17408eea77bcd278 \
    --hash=sha256:4a7cdc2a420ca01058182da4253329764d4bfa055564d1eced90e6ba1e8b1d3d \
    --hash=sha256:4bced6e2a6dba6a28f7dd3c6ce14df1b2dd495923f16ea484cad03decd463b2b \
    --hash=sha256:4cf3468d5ec187ffffcaca8e61929a37448f215dafc1386a12c750a72fe53634 \
    --hash=sha256:4e2c4809c14559aa7ef426f27fb35afbb38104c349a903bf8f3600456764bb38 \
    --hash=sha256:4ed644d75aa94a2baf7ec3a96eaa160ea58c742eb9d27c6506053c5c40fc84ed \
    --hash=sha256:4f6e0852a0283b1b1fd776eeb7b766a5f440b3e2bd31ab51af3b400585f3965c \
    --hash=sha256:5066b244f576f91afc8ee3ba029a89f99d39c79b1853fe9d39bea9f0afbec148 \
    --hash=sha256:5086f9975abb1ab531ee6afca1761e4b59a19b446f3f6522ed776963228cfe5a \
    --hash=sha256:50b5bedc9ed8a94fc8857a42ef4f84a81ea88f8d4f05dc8705fb23ee6d8dcca7 \
    --hash=sha256:52704c5d36eb6dda8866493decd61111fff86244c9b1ad225ca01b9e91e5970f \
    --hash=sha256:55ffd6ce583d97dc71dc92e930324c8c0d25aea7e3ade6ae54ef77cedb096811 \
    --hash=sha256:569d65055d367e3dcdf30c3f41119467b73d9ee9faf332bdf40402644f5ac08e \
    --hash=sha256:57f9947a7e57a081c1e3e0a2dd0d2dcf290a4531450e6f611e30084c222a7295 \
    --hash=sha256:5989cb26b2e1efc6a42216a9f6b5ee495ce5ace2e5b352a9af489976b32d1ee2 \
    --hash=sha256:5c22873ad1f0532ba40fa1727f3c0fc1bbbaab6d373d4cbe3f0dc74b2e2521c7 \
    --hash=sha256:5e8b3d0b18fd623afa12ecb2ce8d8becef69f9b5440c6330c7972200e0bb84b0 \
    --hash=sha256:61631e08084be9e21a8967ec3139c7616ed7c5e9368e05c86d1b39562c8a57b6 \
    --hash=sha256:64511c54db4e4987aef4c41923235927428729e8174c5dba488429be70a998ed \
    --hash=sha256:6669c1bf34080161ce49c589cc512ef24d4c704ac9d2b2d3667f519c60418378 \
    --hash=sha256:672d207103e6b16ca098611b0f9efad6bc00afd47c03d6ef62186495ca677dc0 \
    --hash=sha256:6768d67d1bce64270e0fdc2e69309d68b9b18ae56ddf6c711d168e9d051c2cac \
    --hash=sha256:6a45c3d514f2436064db00d7fc8778d888f0236ebfed649b53d13a59e69ad51b \
    --hash=sha256:6bd9e1788e15bfcf6a9082de42e30387e7b85d211ab21e57a939bb8cfaaf8d96 \
    --hash=sha256:6d2a9efe686f9de00d0d1ea32a4a5a86d558a2277501bd78d964214eab625e59 \
    --hash=sha256:6da83a088f8ef93b2d483a8232a4dbf4d69d3d8496b568a03c56becac43e1808 \
    --hash=sha256:7018d4af1cd272e847aa5917983ab5e83e4f6579f9dbfecd4a79c0ca80b144c2 \
    --hash=sha256:71f88e749ea29f67f21f3b36433c1dc54c7729ed2a6d9e2da2e0d9e0d7b224eb \
    --hash=sha256:737c9c3981998eba27f11786f84fddcbabc74068b72a4a1f454ea02094b57b65 \
    --hash=sha256:73e77980c7207854f00fc4e71fb1626868d5740ab4012623d55c7a99ad122a72 \
    --hash=sha256:799c39bdf5e2f1292fedd3009f7b3c9e760f10b2420cb9638d56920840ff6db8 \
    --hash=sha256:7a83aa6e4805df46fed18e989d3d16f86ef60cb50bbc8d9ce3a6be89165fbf6e \
    --hash=sha256:7d3391b2188d18737cb2fa147028b1096236eaa7e156446c650a489fa2cadc91 \
    --hash=sha256:7e1636da3d8dfc220b6dd10264db5f2b165e4888c4518594898fbe381049af8a \
    --hash=sha256:805c8b84534fa10891890f0e4be39f3a99e94615d93e8836bf9fa1fdca2feeb2 \
    --hash=sha256:811d02d5122171c1941357efd8f9bf4ffe907b7f0a1a4e729a880e4be3f46e3e \
    --hash=sha256:8138eb83940ec7299024d92d4dee45f601b9e6c5ffde9d25f4e35e326203c707 \
    --hash=sha256:83b3944fea42a8400edf92fd1770fb8d0d4f7de651353bd2d8525a92dba69a21 \
    --hash=sha256:849dd2bb0e5e4ab2b71c7191726a4a8d5aa8a610daa584728cbee0b710ddc4ef \
    --hash=sha256:8698d70a8081ee8c090dbb394768b5789a1da8b131b5499f89d071dd3cfaf6be \
    --hash=sha256:8781a792a070cf2bd1b86d3aa943894115faaba6e88122a7bf32d62072742453 \
    --hash=sha256:88d59b473bfb03259722600839af9bbd7fa13a2eb514beefeedb95997882f69a \
    --hash=sha256:8909c2f1c6dd65e054ac4b573a91c8384d1492281e55d82d159d653f7a13adf6 \
    --hash=sha256:8965520ac587c94a4ac48b729be3d8b8de00af39699b17585dfb599babe77977 \
    --hash=sha256:8b5d563170ff8ba3181caa967c99a3c804d1dedb702c7cb93a6a7c32247da978 \
    --hash=sha256:8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581 \
    --hash=sha256:8f0fac8b13d14bb06c68195f849371924ae53dd7b1c00fed24650f704383b692 \
    --hash=sha256:9240187afb63d2f9ddc3e032c670356fe941f6e20662ea168a5dc3f1f317e1b3 \
    --hash=sha256:925f929d6b59a8b3f8b8c6ac363cd0af7eecc81efb3071770b3c6717c450a369 \
    --hash=sha256:9348cbb300d224fe3b89793262cb093504d4ae927004468463f745188a193e4a \
    --hash=sha256:9388003072b95f2f1e3fd908604194d653ba21330d811961a78b7da1a77e9e36 \
    --hash=sha256:9438a2648b2195980cb2dd8e53ed7b8df91319e2d0b70ae61a9e1d1bc8d3bec9 \
    --hash=sha256:94e4c421742086aeee4c32a506eec8859d7634aad943f7e6aacf70f813478768 \
    --hash=sha256:94f5407f7bc64fa6463906b896f9904beeeb7dd8dc116ee8e9056c8714ff9916 \
    --hash=sha256:971a3bbb75d97ae4e2e8f7d4834236f86f85f0c85e04ab2e191db1123b04f80b \
    --hash=sha256:9e227f3dbe6bde7491cf0a9965d00b88c6b1a4a95d11480ddf88bb96d397c19f \
    --hash=sha256:9e25feb9e330b63edb0278a0acdf85e50d0cb0fbf49c3084abbe4e24ae195346 \
    --hash=sha256:9f098115c247e11d138ab83a28fa0323c77015007ea2df73ba5fd714dfefd67c \
    --hash=sha256:a18f38cafc329bac5e3c2b96c765b4c96d3d103421ed22ab7988c1e3fce27464 \
    --hash=sha256:a4bbd2d87dd233b9fc5812160c3d0ffbe42edc22a26ce0469f58479ede633fe9 \
    --hash=sha256:a5fcffb37e602b0b3c1638a97746b9b96125caa9bcf6fa41d337a9261de231ee \
    --hash=sha256:a8e9f292fcda89b324f2f5c91d13f1424a153e40fc2756f38ee23b15835ff300 \
    --hash=sha256:a9f54054101545a9a9cccefddf54316aa6e4491611fcbef9e91b3b6bebec04f6 \
    --hash=sha256:aa2c838cc024642cc04c6854232f32b43e5e22833dd11119c1766c7873b8370d \
    --hash=sha256:ac0c7c9f1609b0c4c114feb1d7a3409564c7fb77e360bed9e97e5d25dfeaf868 \
    --hash=sha256:add96447a86d205ab616665d53b2950ee81083757f56e6ea833c8b2917646b46 \
    --hash=sha256:ae9dcb8fbe244cb82f8a6458b455b927a03685e383d9bacf1ea5ce180b96dc97 \
    --hash=sha256:b4a635a0487774f841cb1fb62e907e7195cc95bc761e053184b8acc3ceb20733 \
    --hash=sha256:b4d12837e0203bbace818ff4a7461afdcd78bcd782351cea148139180d7bcffe \
    --hash=sha256:b61687d0828e72bf5cda24a2690188f37170bd31c9359ac97e4e66569f120a16 \
    --hash=sha256:b807e598953730f82e4eae3bd30f6a122cf6b31c398c6b504c0e04c13c170429 \
    --hash=sha256:b8cd1f918b26fd7b1832ece557cc18f2d8747309ff8b3f0ef9d4250c5ad67a39 \
    --hash=sha256:b91cc9d336957239ff200f30097e6fea2dc6d6fb3c81e853eaa09eac904fd894 \
    --hash=sha256:bd3ce56ae2cbae3ba82b683bc425cd7e48d2ed8b10f3e818186b6f5646d9271c \
    --hash=sha256:be6cb0c799abb0e2ba3e618e6d28ddddf7e485f6c2ce938dfa237daf3905072c \
    --hash=sha256:befb4158af32106b9a93db8d6d1d1cbbd418c0d5aca0cabb7b1780abf0c89169 \
    --hash=sha256:bf053da3c97a4bc5ecfbb218cdd2983febd91c617be8367d139882aa11e490aa \
    --hash=sha256:c02e8f18bdedba082cef725942ac823b9b60656db07f7e265cb31618dfd00d77 \
    --hash=sha256:c1bc67752d5f21013cfe430df4062441714eab79f65a6a05e01505957e9c35fe \
    --hash=sha256:c61750fadcd119d0825bcb7d7d675dd264dcc89cc05292aab5be68ebdbb374ad \
    --hash=sha256:c90d5b3d4e944e065a301d741b3c1d784f6bd1f503aa68b4967e32b2ba313d85 \
    --hash=sha256:c9a7f43c0b202b334cc9184af09bb8f21d3a209e038efaf106936fb69e6b026e \
    --hash=sha256:cb96e6e088d6cf71c1ea977510948320234824cf226e32f6f6e044f7a9c82b34 \
    --hash=sha256:cf63c214fe879a65e69a386f915e36104fc84254ab141240f8854602d8e0be2a \
    --hash=sha256:d1aca03ede943eb80ab3d63bb082c84b7aab85ea83bd0fd0c200260945fb49d9 \
    --hash=sha256:d2e56fd3b00222722abfb3f5f0759ddbae4b90811b5ad4343c64030ad1bde70c \
    --hash=sha256:d5f93ebbeb8032d47e349328ec8662d973d9b05a70b3c35df1f91fe419b84749 \
    --hash=sha256:d882a373d8093c2941e01291b7ced96e9cbe4781da9a7751ca7e6c70385e5214 \
    --hash=sha256:d920abdfa61279ba1a2ef9484aab07bf03331f8c08a10120fa332353d06e6932 \
    --hash=sha256:da2af0d7aebfc2074080d72efa6ab8317c62481ef1f896f65d9999c1c01f4494 \
    --hash=sha256:dd8ea6ebee7aedbf7c749fa80521d9ccf1ba473e0d1e14805caafbaad281c889 \
    --hash=sha256:de8b364c423ef0a4bad9069657d617f9a5d2b2062457a89b1fa16ee199c399c1 \
    --hash=sha256:df1ae86ff54725a01fa1a0510b914ca53a161b7050be74f6204e24aded5971d0 \
    --hash=sha256:dff05cb7016dff1e9fd68f4122c127b65dfc59de5306cfb7ad92f956f230bee2 \
    --hash=sha256:e1a622f13970d81f95d0c72f9dc090dce9085fccfa4c9f2174377ee32bd15786 \
    --hash=sha256:e49fb0d1ce92cfa0cb198cc5b1b11cdf9d0638658e2a2db2687e39db7c87fc78 \
    --hash=sha256:e5c802729725bd07e2bc3ab7b76dc7e0bbfc53129d8f1eb1c002c24cf774717e \
    --hash=sha256:e841068dc0be4cb6dfb5c890eb88cbdcff2f4a332393c7ec94e8e618bd32c1a8 \
    --hash=sha256:e916035e3e9930cbdfdd10abf48861340221857f45509565898e012263f7b289 \
    --hash=sha256:eba154571c16e032112afac0dc2dfe9e63c2ceb7aedd07bb7eecf2ce26d4dd4c \
    --hash=sha256:f03460ff076f70ab595bb45a0205ccea1971443575b6920c52e755dec2b3fbfe \
    --hash=sha256:f0ec3b750b59375eab5b0fb2b9254810c00a3375be6d789899f1055a1d556237 \
    --hash=sha256:f291bcf42ae98eb5107edb162c3c998b4a89648fd8e99ed4cbd12705292788cd \
    --hash=sha256:f61efe1d2fe0de16158a5fe1d1cf3c14bdb6aecd54d8938fd26512c525c1f624 \
    --hash=sha256:f68edfc67aabac33708941f26f22a7b8e9f81429bc0cf249fcf7d66b23af8d19 \
    --hash=sha256:fa95848c929b6a75f6848d3c9793e59db365ee436776e57db835cdbfa79ba977 \
    --hash=sha256:fd9f8797427910198f95bced71ddfed61130d7e349213bfb8466c9c99e2c46a8 \
    --hash=sha256:fdb4ca07ab75ffadab4a8b135ad59cdbb3156b99310f3d565370da74a15d6bd3
    # via jinja2
mdurl==0.1.2 \
    --hash=sha256:84008a41e51615a49fc9966191ff91509e3c40b939176e643fd50a5c2196b8f8 \
    --hash=sha256:bb413d29f5eea38f31dd4754dd7377d4465116fb207585f97bf925588687c1ba
    # via markdown-it-py
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
    # via sympy
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
    # via torch
numpy==2.5.3 \
    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \
    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \
    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \
    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \
    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \
    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \
    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \
    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \
    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \
    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \
    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \
    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \
    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \
    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \
    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \
    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \
    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \
    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \
    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \
    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \
    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \
    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \
    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \
    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \
    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \
    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \
    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \
    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \
    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \
    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \
    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \
    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \
    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \
    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \
    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \
    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \
    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \
    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \
    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \
    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \
    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \
    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \
    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \
    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \
    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \
    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \
    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \
    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \
    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \
    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \
    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \
    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \
    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \
    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \
    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \
    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \
    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \
    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \
    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \
    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \
    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \
    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \
    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \
    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \
    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \
    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab
    # via
    #   molformer-chemistry-pipeline (pyproject.toml)
    #   torchvision
    #   transformers
nvidia-cublas==13.1.1.3 \
    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \
    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \
    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5
    # via
    #   cuda-toolkit
    #   nvidia-cudnn-cu13
    #   nvidia-cusolver
nvidia-cuda-cupti==13.0.85 \
    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \
    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \
    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151
    # via cuda-toolkit
nvidia-cuda-nvrtc==13.0.88 \
    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \
    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \
    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b
    # via
    #   cuda-toolkit
    #   nvidia-cublas
nvidia-cuda-runtime==13.0.96 \
    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \
    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \
    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492
    # via cuda-toolkit
nvidia-cudnn-cu13==9.24.0.43 \
    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \
    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \
    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f
    # via torch
nvidia-cufft==12.0.0.61 \
    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \
    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \
    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3
    # via cuda-toolkit
nvidia-cufile==1.15.1.6 \
    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \
    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1
    # via cuda-toolkit
nvidia-curand==10.4.0.35 \
    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \
    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \
    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f
    # via cuda-toolkit
nvidia-cusolver==12.0.4.66 \
    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \
    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \
    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65
    # via cuda-toolkit
nvidia-cusparse==12.6.3.3 \
    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \
    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \
    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79
    # via
    #   cuda-toolkit
    #   nvidia-cusolver
nvidia-cusparselt-cu13==0.8.1 \
    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \
    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \
    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215
    # via torch
nvidia-nccl-cu13==2.30.7 \
    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \
    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50
    # via torch
nvidia-nvjitlink==13.4.92 \
    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \
    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \
    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \
    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323
    # via
    #   cuda-toolkit
    #   nvidia-cufft
    #   nvidia-cusolver
    #   nvidia-cusparse
nvidia-nvshmem-cu13==3.4.5 \
    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \
    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9
    # via torch
nvidia-nvtx==13.0.85 \
    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \
    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \
    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519
    # via cuda-toolkit
packaging==26.3 \
    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \
    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c
    # via
    #   huggingface-hub
    #   transformers
pillow==12.3.0 \
    --hash=sha256:00808c5e14ef63ac5161091d242999076604ff74b883423a11e5d7bbb38bf756 \
    --hash=sha256:04f01d28a6aaff387bf842a13be313df23ba0597a44f1a976c9feb3c6ff4711a \
    --hash=sha256:06ff022112bc9cbf83b60f8e028d94ad87b60621706487e65f673de61610ab59 \
    --hash=sha256:0740a512dc522224c77d9aa5a8d70d8b7d73fb91f2c21125d8d025d3b8990e45 \
    --hash=sha256:0847a763afefb695bc912d7c131e7e0632d4edc1d8698f58ddabec8e46b8b6d3 \
    --hash=sha256:0dd2064cbc55aaec028ef5fbb60fa47bb6c3e7918e07ff17935284b227a9d2df \
    --hash=sha256:0feb2e9d6ad6c9e3c06effe9d00f3f1e618a6643273576b016f591e9315a7139 \
    --hash=sha256:10e41f0fbf1eec8cfd234b8fe17a4caac7c9d0db4c204d3c173a8f9f6ef3232b \
    --hash=sha256:1182d52bc2d5e5d7d0949503aa7e36d12f42205dc287e4883f407b1988820d39 \
    --hash=sha256:164b31cd1a0490ab6efae01aa5df49da7061be0af1b30e035b6e9a1bfe34ee6e \
    --hash=sha256:1657923d2d45afb66526e5b933e5b3052e6bdea196c90d3abb2424e18c77dae8 \
    --hash=sha256:186941b6aef820ad110fb01fb06eb925374dc3a21b17e37ec9a53b250c6fe2d1 \
    --hash=sha256:1cca606cd25738df4ed873d5ad46bbdb3d83b5cbca291f6b4ff13a4df6b0bbe8 \
    --hash=sha256:21900ce7ba264168cd50defae43cd75d25c833ad4ad6e73ffc5596d12e25ac89 \
    --hash=sha256:236ff70b9312fb68943c703aa842ca6a758abfa45ac187a5e7c1452e96ef72b5 \
    --hash=sha256:23aceaa007d6172b02c277f0cd359c79492bbb14f7072b4ede9fbcaf20648130 \
    --hash=sha256:23d27a3e0307ec2244cc51e7287b919aa68d097504ebe19df4e76a98a3eea5bd \
    --hash=sha256:24870b09b224f7ae3c39ed07d10e819d06f8720bc551847b1d623832b5b0e28d \
    --hash=sha256:251bf95b67017e27b13d82f5b326234ca62d70f9cf4c2b9032de2358a3b12c7b \
    --hash=sha256:25b9b82bb22e6e2b3cd07b39c68b7b862001226cb3dff7130d1cb914121b39ed \
    --hash=sha256:28ce87c5ab450a9dd970b52e5aca5fe63ed432d18a2eaddd1979a00a1ba24ace \
    --hash=sha256:300557495eb45ebb8aec96c2da9c4be642fbf7cd937278b4013ba894ea8eb0eb \
    --hash=sha256:30f2aa603c41533cc25c05acd0da21636e84a315768feb631c937177db558931 \
    --hash=sha256:331b624368d4f1d069149002f25f44bc61c8919ce8ddb3c45bdad8f6e2d89510 \
    --hash=sha256:37d6d0a00072fd2948eb22bce7e1475f34569d90c87c59f7a2ec59541b77f7a6 \
    --hash=sha256:37dc8f7bbb66efe481bb60defacef820c950c24713fb44962ed6aa2a50966de1 \
    --hash=sha256:3b8182a766685eaa002637e28b4ec8d6b18819a0c71f579bf0dbaa5830297cce \
    --hash=sha256:3edce1d53195db527e0191f84b71d02022de0540bf43a16ed734ed7537b07385 \
    --hash=sha256:446c34dcc4324b084a53b705127dc15717b22c5e140ae0a3c38349d4efec071e \
    --hash=sha256:4998562bf62a445225f22e07c896bb04b35b1b1f2eb6d760584c9c51d7a5f78c \
    --hash=sha256:4b0a7fe987b14c31ebda6083f74f22b561fd3739bc0ac51e019622e3d72668c7 \
    --hash=sha256:4e8c2a84d977f50b9daed6eeaf3baef67d00d5d74d932288f02cb94518ee3ace \
    --hash=sha256:4f883547d4b7f0495ebe7056b0cc2aea76094e7a4abc8e933540f3271df27d9c \
    --hash=sha256:514435a37670e3e5e08f3945b68718b6ed329bb84367777e16f9f4dfe1e61a0f \
    --hash=sha256:53aa02d20d10c3d814d536aa4e5ac9b84ca0ff5a88377963b085ad6822f93e64 \
    --hash=sha256:5594fc43d548a7ed94949d139aa1341b270f1863f11cfd37f5a6c8b778a6b67f \
    --hash=sha256:571b9fcb07b97ef3a492028fb3d2dc0993ca23a06138b0315286566d29ef718a \
    --hash=sha256:57b3d78c95ba9059768b10e28b813002261d3f3dfc55cc48b0c988f625175827 \
    --hash=sha256:5afb51d599ea772b8365ae807ae557f18bccfe46ab261fd1c2a9ed700fc6eb17 \
    --hash=sha256:6b02afb9b97f65fbca5f31db6a2a3ba21aa93030225f150fa3f249717e938fb4 \
    --hash=sha256:6c0016e7b354317c4e9e525b937ac8596c38d2d232b419529b9cd7a1cd46e39a \
    --hash=sha256:71d6097b330eea8fd15097780c8e89cb1a8ce7838669f48c5bacd6f663dd4701 \
    --hash=sha256:756c768d0c9c2955feb7a56c37ea24aea2e369f8d36a88da270b6a9f19e62b5e \
    --hash=sha256:78cb2c6865a35ab8ff8b75fd122f6033b92a62c82801110e48ddd6c936a45d91 \
    --hash=sha256:7a743ff716f746fc19a9557f60dab1600d4613255f8a7aeb3cdde4db7eb15a66 \
    --hash=sha256:85f998ea1848bc6757289e739cfbdda3a04adfd58b02fc018ce54d754a5ce468 \
    --hash=sha256:8728f216dcdb6e6d555cf971cb34076139ad74b31fc2c14da4fafc741c5f6217 \
    --hash=sha256:877c3f311ff35410f690861c4409e7ccbf0cd2f878e50628a28e5a0bb689e658 \
    --hash=sha256:8cd2f7bdda092d99c9fc2fb7391354f306d01443d22785d0cbfafa2e2c8bb418 \
    --hash=sha256:8e95e1385e4998ae9694eeaa4730ba5457ff61185b3a55e2e7bea0880aef452a \
    --hash=sha256:962864dc93511324d51ddbb5b9f8731bf71675b93ca612a07441896f4688fb8c \
    --hash=sha256:9cf95fe4d0f84c82d282745d9bb08ad9f926efa00be4697e767b814ce40d4330 \
    --hash=sha256:9e881fca225083806662a5c43d627d215f258ff43c890f831966c7d7ba9c7402 \
    --hash=sha256:a2b55dd6b2a4c4b7d87ffa56bdb33fdc5fdb9a462173861a7bc097f17d91cb09 \
    --hash=sha256:a45650e8ce7fafffd731db8550230db6b0d306d181a90b67d3e6bca2f1990930 \
    --hash=sha256:a876864214e136f0eb367788dbd7df045f4806801518e2cfe9e13229cfe06d8f \
    --hash=sha256:ae26d61dfa7a47befdc7572b521024e8745f3d809bd95ca9505a7bba9ef849ec \
    --hash=sha256:af8d94b0db561cf68b88a267c5c44b49e134f525d0dc2cb7ed413a66bc23559a \
    --hash=sha256:b343699e8308bdc51978310e1c959c584e7869cc8c40780058c87da7781a1e94 \
    --hash=sha256:b3c777e849237620b022f7f297dd67705f9f5cf1685f09f02e46f93e92725468 \
    --hash=sha256:b629de27fda84b42cde7edef0d85f13b958b47f6e9bbcbba9b673c562a89bd8b \
    --hash=sha256:ba09209fbe443b4acccebe845d8a138b89a8f4fbaeedd44953490b5315d5e965 \
    --hash=sha256:ba54cfebe86920a559a7c4d6b9050791c20513650a1952ebe3368c7dc70306f8 \
    --hash=sha256:bcb46e2f9feff8d06323983bd83ed00c201fdcab3d74973e7072a889b3979fcd \
    --hash=sha256:bcc33feacfaefce60c12fd500a277533bdc02b10a19f7f6d348763d8140bbba7 \
    --hash=sha256:bf16ba1b4d0b6b7c8e534936632270cf70eb00dbe09005bc345b2677b726855c \
    --hash=sha256:cf1845d02ad822a369a49f2bb9345b1614744267682e7a03527dc3bf6eea1777 \
    --hash=sha256:d69141514cc30b774ceea5e3ed3a6635c8d8a96edf664689b890f4089111fb35 \
    --hash=sha256:d9c7f76c0673154f044e9d78c8655fb4213f6ca31a836df48b40fe5d187717b9 \
    --hash=sha256:dbce0b29841537a2fa4a214c2bbf14de3587c9680caa9b4e217568472490b28f \
    --hash=sha256:dc624f6bc473dacdf7ef7eb8678d0d08edf15cd94fad6ae5c7d6cc67a4e4902f \
    --hash=sha256:e158cb00350dc278f3b91551101aa7d12415a66ebf2c91d8d5ac14e56ddd3ad0 \
    --hash=sha256:e491916b378fba47242221bb9ead245211b70d504f495d105d17b14a24b4907c \
    --hash=sha256:e795b7eb908249c4e43c7c99fac7c2c75dab0c43566e37db472a355f63693d71 \
    --hash=sha256:e7e480451b9fa137494bccd3a7d69adbe8ac65a87d97be61e11f1b1050a5bac3 \
    --hash=sha256:e91206ee562682b51b98ef4b26a6ef48fd84e15fd4c4bc5ec768eb641d206838 \
    --hash=sha256:e9871b1ffbfa9656b60aeee92ed5136a5742696006fa322b29ea3d8da0ecc9cf \
    --hash=sha256:e9aeb04d6aef139de265b29683e119b638208f88cf73cdd1658aa07221165321 \
    --hash=sha256:ebaea975e03d3141d9d3a507df75c9b3ec90fa9d2ffd07567b3a978d9d790b26 \
    --hash=sha256:f0606c8bf2cdefea14a43530f7657cbbb7ecf1c4222512492ef4a4434a9501ec \
    --hash=sha256:f13c32a3abd6079a66d9526e18dad9b6d280384d49d7c54040cd57b6424041d9 \
    --hash=sha256:f7401aebd7f581d7f83a439d87d474999317ee099218e5ad25d125290990ba65 \
    --hash=sha256:fa4ecea169a355be7a3ade2c783e2ed12f0e40d2c5621cda8b3297faf7fbb9f5 \
    --hash=sha256:fbd139c8447d25dd750ab79ee274cc5e1fe80fc56340ab10b18a195e1b6eca3e \
    --hash=sha256:fdafc9cce40277e0f7a0feabce0ee50dd2fa1800f3b38015e51296b5e814048d \
    --hash=sha256:fe3cca2e4e8a592be0f269a1ca4835c25199d9f3ce815c8491048f785b0a0198 \
    --hash=sha256:ffd0c5368496f41b0944be820fcb7a838aa6e623d250b01acf2643939c3f99d7
    # via torchvision
pygments==2.21.0 \
    --hash=sha256:2363c69b61c4a97c838da3b130dcd6468f4848992b21a82f2a63ec34377137d9 \
    --hash=sha256:610ca751c9bc2492b38eb9a38a7fbc93edbbb2d7182edaf34e66ae493dee5c8c
    # via rich
pyyaml==6.0.3 \
    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \
    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \
    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \
    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \
    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \
    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \
    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \
    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \
    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \
    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \
    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \
    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \
    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \
    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \
    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \
    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \
    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \
    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \
    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \
    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \
    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \
    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \
    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \
    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \
    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \
    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \
    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \
    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \
    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \
    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \
    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \
    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \
    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \
    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \
    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \
    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \
    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \
    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \
    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \
    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \
    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \
    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \
    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \
    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \
    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \
    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \
    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \
    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \
    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \
    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \
    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \
    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \
    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \
    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \
    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \
    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \
    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \
    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \
    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \
    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \
    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \
    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \
    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \
    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \
    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \
    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \
    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \
    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \
    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \
    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \
    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \
    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \
    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0
    # via
    #   huggingface-hub
    #   transformers
regex==2026.9.29 \
    --hash=sha256:01000ddf0e3ffef97f2413ceb514f6313040106b6d18a03ee00a4fe35c1eb1db \
    --hash=sha256:0166844493626c5015c6088ee15c9ca2fd060ca15b7641d1657da6a58432ae33 \
    --hash=sha256:044265d77d94f5e3cb2fd72c76723807c429cb8c533e9d4672d0334a6f14f588 \
    --hash=sha256:0476e5bcbe6e1ba3d1c4cc7bbb1c3ba78e3b979b5c8a88d0a6a8cdd4992b8c84 \
    --hash=sha256:066d0e3dbfdd739bce2bf8c2a41dd16f73e3d8adc2eb06dd803a36a307f56075 \
    --hash=sha256:0b65c72739f981377c9c22e0c5c3cd7f42da7bd8a3c9209330fac772c7d893ed \
    --hash=sha256:0c992c19cd45058a4b92f68f139c93db168b48fb1f322c9a7cd620806afb6b51 \
    --hash=sha256:0cc63b5e47c12a48d90c7e9d7de6a035dd14f62868aaedbb4e0ff8ba2b8bfe7b \
    --hash=sha256:0dd8af32e9f7b56b7f95cc1fd79b23054c3bdc172392ae560acc24d57b7ffe71 \
    --hash=sha256:0def9fb6abac55492d6d51cddb7225d07d6f279e774e0adc08569a54a5fc8d46 \
    --hash=sha256:0fd2c901cc307a745ad4bc87f20060d7a0825a3371d1e93488af22e7a387f78f \
    --hash=sha256:1043aedf5917caa861bcb25a9c11460049656bdf0017a90a309fa8f255467725 \
    --hash=sha256:121a76a0985db80ceae9e171c337f8c927868e37d01b54e3ce87bc87f9c6a208 \
    --hash=sha256:143533cc4b6fbc5b95aca0a5b8d541088d374831593def000ec89322c220221d \
    --hash=sha256:14e953ff3607c92d7675bf79c4d4509ef6782aa8c08509f179f9b3d6d0679e86 \
    --hash=sha256:18ae8eed4526e35bdb754d61562b90bf5c00a67fdcf3cc1380dd59597486631b \
    --hash=sha256:19959129885356df0e97556856f77eb2888380dac18bed075a7c05c5128c618d \
    --hash=sha256:1ba8c6a416569ce0d37e83e28a254a61dc99a419084dfb6476cea02d997f74fa \
    --hash=sha256:1c2a0026062abcc321a53db4a185ceba0b59a66b5d37b0808917a88b55a5257f \
    --hash=sha256:1d9fe8091b2e89d470df68a9331111ed008ae8aae6bf1e8e1fba4086a495c84e \
    --hash=sha256:2089fe39c406784d90101c726755ffa1497bb74638fd434300d2b88006186de8 \
    --hash=sha256:23ae6fdad9e63e54038f5ef78aba2933faca61e24d432786589e737bc5522ebb \
    --hash=sha256:26ec4ccce55aa533fbd603d08911b01101a8fcfec987845ac3ae2c7087b2bde3 \
    --hash=sha256:2f7f7aa47b229f2b39a2ae2596d2ad5625d77b5eb9856fac2dab3eb506cdd0a0 \
    --hash=sha256:31b003f9a070335e2a8233ee9b14a3ca8e6d792012ae011f741bf0aaf11744c5 \
    --hash=sha256:32ab11df9677ca80bcbb5fe4eb1da9109a5019239a054836efc6fa1c64e683cf \
    --hash=sha256:33026515aebc0e70d1c89978e53e8d695d35d9e472f8d5b34465ba3c74028650 \
    --hash=sha256:34b6925af9853bf461950e6508910f179fd6e9b1a7ec8548e069606b7e51a26b \
    --hash=sha256:352cf115a810b357caa35193ab656ecf5ef41056855e82f292c99e8514f8d954 \
    --hash=sha256:39ab5894d971f9ac68baa6eca5c50387db579cfcacf36ae8df3feceb1815e6d0 \
    --hash=sha256:3a21a9509d0ee88e7a70e1ad228cd2f0e0fd1e187458db132e8a8d18c97daf9d \
    --hash=sha256:3c5c2ef13797466aa64170cbb66ad98a32351dd4127694cea7199f80f213750d \
    --hash=sha256:3e778bfccd63075167709136afbc251c1f683758d5bf49c803c60ac3f894ce6b \
    --hash=sha256:3f1e6cb402a89457582cd696f982559217d13484a193202c394015297968c86d \
    --hash=sha256:42e82e578c904445d4c8a35b8f28052cf567593215fa5db06266fbc6f77aaa2e \
    --hash=sha256:4408b2b27a95ca8cc48b7411945753773353b5c93b307754781086c99d3a576f \
    --hash=sha256:446654b29bfaa30500d80947eda42cef1449dc8a87f4e3cf061cc8485d3a1f0b \
    --hash=sha256:45010bcfe66df41522d56c9b6114e87ecc597a08970ff6a2ced24415c141ae5f \
    --hash=sha256:49ee178ca31c94621294bf9b8b676a92a2e6bba8af0529591753719e57edb621 \
    --hash=sha256:4d7d93613b01b0199961330e49cfc52d479b3d5776c56c691db31130c0a07d91 \
    --hash=sha256:4fb41211d2333eb930a51e0546a65999761cf1f572a4da56ef9b8a62966c06f2 \
    --hash=sha256:4fe97894d1b306c919b4e50def1e6f6c522f4d03a7283811f4d108f1ce5d3ac2 \
    --hash=sha256:554bffadcbcb6d5f4e5fb10a61cc52084b9a63d1dab5f10bcd2c4343972e8e2c \
    --hash=sha256:59b49507f47479e299a9e1bc41b5cb83a7afda0540625f1dbae886615978acbf \
    --hash=sha256:5eeb8edc6110d9194a4d0d54610f64c37a31c605b5dbb7e407fc6ec7fa34a4a1 \
    --hash=sha256:612b709381c0355b70d89cdb51b7f670591ed5cbbc0e3b5337488019dc667b65 \
    --hash=sha256:61956f074ecd123f55adca68ee3eab46e6a07ad3f8e64e6db95dfacb444f55c4 \
    --hash=sha256:6398d5145689503412cc1748895242598d8846b8967b851133b20dc2ed1e21e8 \
    --hash=sha256:65b408d8fcb273e3499e7ef2ce796810da1becd208c7fb4373692a242d79d461 \
    --hash=sha256:686ac5350fceae63830bb98805fcb8039325bf4c06d9f6f048ff65229d5bffa5 \
    --hash=sha256:6a1a824fbed817e0a891103886b68f063b1e83cc51bc97192a90a60195a9291f \
    --hash=sha256:6abb75ab16bc3281714a5b99548a2225db70dba1f995f6d7f7419b76eb5a8fbe \
    --hash=sha256:6f7121a8914ed13fcfe2099f895341bfb789f004d4c5a0bdece8fa667da10849 \
    --hash=sha256:7020ed44df30b3aa492c00ee3b52d0548c1f30c2c6c5bb13ae897680900d3413 \
    --hash=sha256:720537c7ea6f80dc61913184edb0ce2497a306b39ef19f28505b322553d52bdb \
    --hash=sha256:724184b4aafed865e4f13ca313fdcb43024300c028ec67319cfa16847d84685e \
    --hash=sha256:7c03031610e3e6ed1768a2b7a8fc84637c1257b50c5eacaf094c6e17a84fc563 \
    --hash=sha256:80a5ea3b4fd9d6a5b9a44f7976a9acaaab35aa3c1f6b29e5bd857dfabaded223 \
    --hash=sha256:80c7cadd3fd2bfde5df8aa0787e315812cad0c313a753095d02f4c2b6c01677b \
    --hash=sha256:80ea96f5c1a30bf09007d48466521d9c294bebe197c708c3359096e3e3691632 \
    --hash=sha256:864e9b87ac33c3fb9fb4ad48166d4fdb579c351d5c77deb0d34bccb36a775cd9 \
    --hash=sha256:87fb80cbe3557e27e7b28b995c2b2eedf689b8886f941ab93e0e288f0976518a \
    --hash=sha256:8873c4a11c50b9989168881aeb3f08859f469d809941866aa1feefd8be5431f6 \
    --hash=sha256:888d60953908dcf761aa320c3e390ab8556efbdb551ace63921de90f6ae0848d \
    --hash=sha256:8b5fcc4771732191b2b7d1dd68d8f0353f47f8d90b6150f6dce58bf1112442cb \
    --hash=sha256:8f39588af4731c8923c26810eb3b33f76f17633985e40f59c3cd45a33805a895 \
    --hash=sha256:9173db3be74a35cb6731701094b98120f7ee4876a287882a59cdea1fa7da342f \
    --hash=sha256:92f05c9c42bde5785dc48770bc2194d9f7442544156f951e19cd31b096cec562 \
    --hash=sha256:951733b1bbdb71e377cec567b409f1a7881b47cfcad84121aa74cb575fa425ea \
    --hash=sha256:957bb708e8057ab1649ba566456429d691ec9b90d1c9ad1af1ba7ffbbeaf05f2 \
    --hash=sha256:9916fda742cd4eede63b286f58c06718324265d727ce0856eb1aac86d0d150d6 \
    --hash=sha256:9e1d3a4cb7993b708f0ada8d0c84590efd853f169e7147d2202c9da503180242 \
    --hash=sha256:9e4482589065c8ecd761cff522dcd85f2d39e62f551e37e025d1c7d54772def3 \
    --hash=sha256:a5300757f8a68f5b6cc33f57338d72a0e3589c5cc9ad5f8504ea06f028be582a \
    --hash=sha256:a540abfab208e1b7ef2df231c40ef3b6cbb30a0aad6204e9b6a81c10a6794628 \
    --hash=sha256:a5758353650079898dc1b2b0e95aa51fa23a30d020e06f62c430dd08ee56cdd8 \
    --hash=sha256:a64b85a4760337cfefdb27d42da6ed8b58e8cde3f2d57b6ef43e76ef6ea9ef47 \
    --hash=sha256:a655d34b2a6943af32401f3d94f72e9d731f6ad16285815550bf2b4ee69d420a \
    --hash=sha256:a714befaacbd10092ffe4cea0d3c5f008fb9efe9bc322c715bcdfdee414b9a3d \
    --hash=sha256:a760da040b47767b4b873adfb7c3b691e9ba2fc60f113f9d0b88f1a62f323e85 \
    --hash=sha256:addd736a0547d553283adaf4e05d7104e7f2c7b0b092e9b4d28756825f14531f \
    --hash=sha256:ae4613d7d9dda60fcba95f846cc6f808017f1843f392cf9daad14a6534493d71 \
    --hash=sha256:b11b589e00095ec69cf79841a76360f9b079e95b0368a25b5ebb951ab0c157ff \
    --hash=sha256:b3e445b66c80b4eb4234e855ce94d9adc183eedbd632816228d89930b91b2c5b \
    --hash=sha256:b7b893976e7fe42053da64f2aa27239c24252fd2ec6df471e1be197c0addc3b1 \
    --hash=sha256:b84f186a7f0536fe4ff9a9fa12d06d007b9b71d4b5352ddcc41f59ad6522a312 \
    --hash=sha256:b89efc38431793d28b7cd91227e2f952ad7c48df19132b17f43a5fec3c14143b \
    --hash=sha256:b97a38fb4c732b6832db6bf108963adbcd82ef1268ba2025dce390f45af75efa \
    --hash=sha256:b9d74e4eee9ddb64c2e92d5d61472c59c21684c059eb7b68767be9628e977859 \
    --hash=sha256:bb90e7177944b6684738c1fc36aabd2dd00d1de3be7dbe09f91e196f1bc0dc81 \
    --hash=sha256:bec37990e3d6121f29ecfb594bd8f1bf009e9f7926daba2e50e3b27d3892a783 \
    --hash=sha256:bf3c49863c23a1ad6da9c30351aed6cff8d5ddbeb63c5c8420ae54e98c7d0138 \
    --hash=sha256:bf48516e35cf848390ea68850aba53e7c333720d2945b4d2c25b69fc5171723f \
    --hash=sha256:bfc71e6d970419c1309b3640305298643e2a734cad3f7cfb6d2ddee4175ab53d \
    --hash=sha256:c0094897d7d01f184b2d7fe8c56c66d64efe01b31f4b7d34205b391387df1111 \
    --hash=sha256:c03c6eb6ece86dfdcbb34799efaa339b093132e1aceed491ba5e08fe06cdf699 \
    --hash=sha256:c1a9a6651197fbed6f0212591418b9def774fc3f8324f78d1bf0e6a63e5f8aa1 \
    --hash=sha256:c3589f40749acce747510bf5d589d54e376cb0930ea58b35effac97e5312b0c1 \
    --hash=sha256:c4e38dd8f39c43a91d2410ad2b85610701b0979342c3df1d69eaf8e838c757d8 \
    --hash=sha256:c6c8fabf1dafc1f1ddcbb67896d3f93efb092e8c4b6322d7389b944e76a484e5 \
    --hash=sha256:c90fcf7804ea0a54b896ce0f2b9565350220b8d4890fd0db461a476a4c687963 \
    --hash=sha256:c9b602fae1e00b7c035d661ce85575365719192a7b46784bd71cf64c68053aa0 \
    --hash=sha256:ccb64d887a9db1cd76dbc0f92051a1a478a2a67e7f56c62d915cb881d7734704 \
    --hash=sha256:d06fcdecc10fc7954d7c8f27a03c96055fe525274dc84a7b0dbdc3d6b9e03dab \
    --hash=sha256:d0c3082bf79bcd6a614d55916590ad4b8f93200e10b97f463ea5d9d07c9b5f23 \
    --hash=sha256:d49c18f1ea294cf4adde2e5ac256e98c82ea9d708462ce4bf799dffa7cfe8a2c \
    --hash=sha256:d60030baaa7bfbb02d650c126cdcddcb6e33dbff14d819434c8fa2fdcaeeeba5 \
    --hash=sha256:d7cab119d0df0b9413f106b4d7fc34f2872d3574ed3806fb48959c830b1537da \
    --hash=sha256:d9b77b25b4f395f92de6099ab08e8ae2bc7e51dfe157f22900902243a5cc90c7 \
    --hash=sha256:dabee8f4935e731fb46b2a3091bdda0d3d94b3bbfb907d2b4f12eefce4009619 \
    --hash=sha256:db5e82ba15c142425b8406690032df89e39cca4a2e8afbbb9a3d84edc2373ac3 \
    --hash=sha256:dc79d36d0618752265f0d575915bdc5c5130ecb9c9f6b3bcefeae32e4bdfafcf \
    --hash=sha256:ddfa987262763c3c22a8367d2a49c244b018a74c3a8e3ab1a864119ad45c5633 \
    --hash=sha256:e1172147d28d8fbcf8cb8d26c41506169f5ad8fe9ec969cb116835a19d4d8eca \
    --hash=sha256:e11edba5bc344a32b029a7af9d4b3173982dd79eeafa0b9dbd787364414b0509 \
    --hash=sha256:e2c89e9b762c57f59d5e99ee8b20202adb892e35f8d3485741340999ca55058e \
    --hash=sha256:e31f72490b7c12f7790e1e25c3afffd20503ee1bfb43461d7838b871ff244b19 \
    --hash=sha256:e8c65ef3862a8ad6e86492b6ed9327805dd66904c012bd3649dc67d822ed6c34 \
    --hash=sha256:ebb8912f565b8cdbbf27debfe00df04202c20e2f651b9e32767930c5eace3621 \
    --hash=sha256:ed511a0708e2297e1d6431e7fb217e3402791e491e02da800658ace4973df1bb \
    --hash=sha256:edf06545875f3efa31560d94121e95c7fd70d98b1dfedc0157097d79b13b52ea \
    --hash=sha256:f0fe9834e5aeccaf19a0d8feb296d66a24be1a7c9922002f842a682cd5abb787 \
    --hash=sha256:f1a0d5117230dd46b399a30a38afa44f79c99f3168988fdc4f425c3f928b39df \
    --hash=sha256:f37964e4a5e993d2fd45147741e9dff7f34a2d8c00ab94c4ea0514a4677f959e \
    --hash=sha256:f57dc6b8fef170f105d2cf5cdce254f47b137d7755086cf7050f47e16582abba \
    --hash=sha256:f93bc1c3486ef3747e07c9d7c1d0a147b8fbaab975f80e348aed6f71309dfaca \
    --hash=sha256:fb00027a09a8f9f08028b40dce4c933cf73e4833240ed356583fdc9cfa721566 \
    --hash=sha256:fb99cc9d45f48895d9d67f6a0b8a57f08d39c174d9f25ad97a313e0470267b1c \
    --hash=sha256:fdd88ed5e20b1bcdd234421e454962c971aa44b653bdb7f1ea9ef683e90fb649 \
    --hash=sha256:fe3fa1dd453ed5c7f5ea23a26218329790ed7197a99b90e94330e313959a7f52
    # via transformers
rich==15.0.0 \
    --hash=sha256:33bd4ef74232fb73fe9279a257718407f169c09b78a87ad3d296f548e27de0bb \
    --hash=sha256:edd07a4824c6b40189fb7ac9bc4c52536e9780fbbfbddf6f1e2502c31b068c36
    # via typer
safetensors==0.8.0 \
    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \
    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \
    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \
    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \
    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \
    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \
    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \
    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \
    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \
    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \
    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \
    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \
    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \
    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \
    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \
    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \
    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774
    # via
    #   molformer-chemistry-pipeline (pyproject.toml)
    #   transformers
setuptools==84.0.0 \
    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \
    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73
    # via torch
shellingham==1.5.4 \
    --hash=sha256:7ecfff8f2fd72616f7481040475a65b2bf8af90a56c89140852d1120324e8686 \
    --hash=sha256:8dbca0739d487e5bd35ab3ca4b36e11c4078f3a234bfce294b0a0291363404de
    # via typer
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
    # via torch
tokenizers==0.23.2 \
    --hash=sha256:12f0835dc2ee694746a76adf7b1567d4346a4a502ebe93fb1f5f80ea49799b78 \
    --hash=sha256:2e96f5699d5249c9c64aa8412e044f727aae3a4098cf830f9901ec1afc361cde \
    --hash=sha256:325fee2e0418a9dc6c9ecf736a5f5f0db7875183ace9549ae339da76f7a1fbb7 \
    --hash=sha256:41c2f84d172449b4dadb9cdc508e3e364076613c35b16e76ecfe47a60d1e3305 \
    --hash=sha256:43e4f2071e3cc8d5d86421c874aebc82659bb51a68bcdef5a0da75ee89511ccb \
    --hash=sha256:5c56bda1511921587789163e524d196ed8284174ac23abd7685d5ea8da6c4718 \
    --hash=sha256:7b7e37ba198f24150f523e1242e83c4970de4a525480586be5dcc24d9add32c5 \
    --hash=sha256:7f0f085686b9de0d0079e6f874ae053600db64c5d13049e0bbc0119926d25aac \
    --hash=sha256:85a9a357a3764aecc904ee76bdaf8cf1ad8e5a67a1b929a487c4a39b49ed0e90 \
    --hash=sha256:950d7c9426fa72406a0ffeacdbc0bb9985f5db20eb8b263f29c79aaf83105703 \
    --hash=sha256:986670e43691469dcee610ea0f846f91a8f84e91fc6f7a48d4c064414c0ec2bf \
    --hash=sha256:a37039b5dfc4af84eb3ef0a92f4307e28936c8f9adccba2629d36f652e9bf7a2 \
    --hash=sha256:bef235815a067b2648caf6dcc7a71091b0b0fff9ee8057f6451eb9335fae52ef \
    --hash=sha256:debf978920d93ba9c219bd67cc4bbfaf912c9039e41e7a28b91ec15e3728c95a \
    --hash=sha256:e49c394456dd9985787fec76132438ba3fb8911f857b1bf3d40119f9292d41aa \
    --hash=sha256:eb2f9c8a24da020ea8c11a01a19c1c2547912d92121ae4a01cfbca46125dee40 \
    --hash=sha256:f486f402f6f9abee5bb032553736813af0c710a86b2e0ca592634c55cea1f835
    # via
    #   molformer-chemistry-pipeline (pyproject.toml)
    #   transformers
torch==2.14.0 \
    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \
    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \
    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \
    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \
    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \
    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \
    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \
    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \
    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \
    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \
    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \
    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \
    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \
    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \
    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \
    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \
    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \
    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \
    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \
    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \
    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \
    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \
    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \
    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963
    # via
    #   molformer-chemistry-pipeline (pyproject.toml)
    #   torchvision
torchaudio==2.11.0 \
    --hash=sha256:00e9f71ab9c656f0abdb40c515bd65d4658ab0ad380dee27a2efd7d51dabd3d6 \
    --hash=sha256:13cff988697ccbad539987599f9dc672f40c417bed67570b365e4e5002bbd096 \
    --hash=sha256:1424638adb8bb40087bc7b6eb103e8e4fe398210f09076f33b7b5e61501b5d66 \
    --hash=sha256:1a07ec72fd6f26a588c39b5f029e0130d16bb40bc4221635580bf8fb18fcbc80 \
    --hash=sha256:1be3767064364ae82705bdf2b15c1e8b41fea82c4cd04d47428a8684b634b6ed \
    --hash=sha256:1c1101c1243ef0e4063ec63298977e2d3655c15cf88d9eb0a1bd4fe2db9f47ea \
    --hash=sha256:478110f981e5d40a8d82221732c57a56c85a1d5895fb8fe646e86ee15eded3bd \
    --hash=sha256:492dd64645e9d0bb843e94f1d9a4d1e31426262ffc594fafecc1697df9df5eb9 \
    --hash=sha256:5847fe2022b17c6580aeb39c8797a443411cc09edfd9183cd50ac1a3b8ccf97c \
    --hash=sha256:6503c0bdb29daf2e6281bb70ea2dfe2c3553b782b619eb5d73bdadd8a3f7cecf \
    --hash=sha256:67f6edac29ed004652c11db5c19d9debb5d835695930574f564efc8bdd061bba \
    --hash=sha256:6ebb59c694909eccb5d61b7cc199d297692012c43286e36d92983aa7bad7586d \
    --hash=sha256:73dab4841f94d888bc7c2aed7b5547c643edc974306919fe1adfb65d57cccf4b \
    --hash=sha256:79fb3cb99169fd41bd9719647261402a164da0d105a4d81f42a3260844ec5e79 \
    --hash=sha256:7e2da1df4f6fe885c46db350a0dc90a0dff4b54541dff8846faa904d255e2bfe \
    --hash=sha256:88fb5e29f670a33d9bac6aabb1d2734460cf6e461bde5cdc352826035851b16d \
    --hash=sha256:986f4df5ed17b003dc52489468601720090e65f964f8bebccf90eb45bba75744 \
    --hash=sha256:9fe3083c62e035646483a14e180d33561bdc2eed436c9ab1259c137fb7120b4a \
    --hash=sha256:a1cf1acc883bee9cb906a933572fed6a8a933f86ef34e9ea7d803f72317e8c1b \
    --hash=sha256:b034d7672f1c415434f48ef17807f2cce47f29e8795338c751d4e596c9fbe8b5 \
    --hash=sha256:bb59ba4452bbbe95d75ad3ef18df9824955625f36698ce9a5998a4a9f3c1ba1d \
    --hash=sha256:bc653defca1c16154398517a1adc98d0fb7f1dd08e58ced217558d213c2c6e29 \
    --hash=sha256:bda09ea630ae7207384fb0f28c35e4f8c0d82dd6eba020b6b335ad0caa9fed49 \
    --hash=sha256:be7ad472acb16d16e98c005f0219b0db06a47dfe8f7b4d177062e1638f871e3b \
    --hash=sha256:cc09cd1f6015b8549e7fe255fb1be5346b57e7fee06541d3f3dbb012d8c4715f \
    --hash=sha256:da2725e250866da42a12934c9a6552f65a18b7187fd7a6221387f0e605fb3b96 \
    --hash=sha256:e3f9696a9ef1d49acc452159b052370c636406d072e9d8f10895fda87b591ea9 \
    --hash=sha256:ed404c4399ad7f172c86a47c1b25293d322d1d58e26b10b0456a86cf67d37d84
    # via molformer-chemistry-pipeline (pyproject.toml)
torchvision==0.29.0 \
    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \
    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \
    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \
    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \
    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \
    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \
    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \
    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \
    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \
    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \
    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \
    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \
    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \
    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \
    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \
    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \
    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \
    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \
    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \
    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \
    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \
    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \
    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \
    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e
    # via molformer-chemistry-pipeline (pyproject.toml)
tqdm==4.70.1 \
    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \
    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4
    # via
    #   huggingface-hub
    #   transformers
transformers==5.17.0 \
    --hash=sha256:78ec1ce21579b38dfb83950a0658cd119f87212a2fcfdff478096ce9d6c03801 \
    --hash=sha256:a153be279169b55b92d8000bf4af294aed684503d091cca7804da2dd8a9de000
    # via molformer-chemistry-pipeline (pyproject.toml)
triton==3.8.0 \
    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \
    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \
    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \
    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \
    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \
    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \
    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \
    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \
    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \
    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \
    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \
    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3
    # via torch
typer==0.27.2 \
    --hash=sha256:269b7eb9d3c202ca84b4bc9618cb04ebb43d3d4d1e567e4c768607232c05f945 \
    --hash=sha256:b3a5fc4342d5fc8fda8fc3010b1cf117e9249aab7fae800c2eff62fd3842d97d
    # via transformers
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
    # via
    #   anyio
    #   huggingface-hub
    #   torch
'''

SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
# One environment per lock digest: a re-run of this cell, or a second Run all in the same runtime, reuses a complete
# environment built from this exact lock instead of rebuilding it; a different lock gets a different folder.
ISOLATED_ENV = Path(os.environ.get('DIMER_ISOLATED_ENV', 'dimer_isolated_env_' + LOCK_SHA256[:12])).resolve()
ISOLATED_PYTHON = ISOLATED_ENV / 'bin' / 'python'
ISOLATED_TOOLS = ISOLATED_ENV.with_name(ISOLATED_ENV.name + '_tools')
ISOLATED_READY = ISOLATED_ENV / '.dimer-lock-sha256'


def _isolated_environment_ready():
    return ISOLATED_PYTHON.is_file() and ISOLATED_READY.is_file() and ISOLATED_READY.read_text(encoding='utf-8').strip() == LOCK_SHA256


if SKIP_INSTALL:
    print('DIMER_NOTEBOOK_CI_PREINSTALLED=1: the pins are already installed; the notebook runs in this kernel.')
elif _isolated_environment_ready():
    print({'isolated_environment': str(ISOLATED_ENV), 'reused': True, 'lock_sha256': LOCK_SHA256[:16] + '...', 'locked_packages': LOCKED_PACKAGES, 'kernel_python': platform.python_version()})
else:
    if platform.system() != 'Linux' or platform.machine() != 'x86_64':
        raise RuntimeError('This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle or Linux Jupyter): its locked environment is built for manylinux x86_64.')
    setup_started = time.perf_counter()
    if hashlib.sha256(LOCK_TEXT.encode('utf-8')).hexdigest() != LOCK_SHA256:
        raise RuntimeError('The carried lock does not match its digest: regenerate the notebook from the repository instead of editing this cell.')
    ISOLATED_TOOLS.mkdir(parents=True, exist_ok=True)
    lock_path = ISOLATED_TOOLS / LOCK_NAME
    lock_path.write_text(LOCK_TEXT, encoding='utf-8', newline='\n')
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2**attempt)
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError('The pinned uv wheel failed its size/SHA-256 check: refusing to run it. Run this cell again; if it repeats, the download is being altered.')
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(name for name in archive.namelist() if name.endswith('.data/scripts/uv'))
        uv = ISOLATED_TOOLS / 'uv'
        uv.write_bytes(archive.read(member))
    uv.chmod(0o700)
    # uv gets no kernel Python path; the managed interpreter is downloaded once and reused on a re-run.
    uv_env = dict(os.environ, MPLBACKEND='Agg')
    for name in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP'):
        uv_env.pop(name, None)
    if not ISOLATED_PYTHON.is_file():
        subprocess.run([str(uv), 'venv', '--quiet', '--managed-python', '--python', MANAGED_PYTHON, str(ISOLATED_ENV)], env=uv_env, check=True)
    isolated_version = subprocess.run([str(ISOLATED_PYTHON), '-c', 'import platform; print(platform.python_version())'], env=uv_env, check=True, capture_output=True, text=True).stdout.strip()
    if isolated_version != MANAGED_PYTHON:
        raise RuntimeError(f'{ISOLATED_ENV} holds Python {isolated_version}, not {MANAGED_PYTHON}: delete that folder (or start a fresh runtime) and run this cell again.')
    ISOLATED_READY.unlink(missing_ok=True)
    subprocess.run([str(uv), 'pip', 'install', '--quiet', '--python', str(ISOLATED_PYTHON), '--require-hashes', '--only-binary', ':all:', '--index-url', 'https://pypi.org/simple', '-r', str(lock_path)], env=uv_env, check=True)
    ISOLATED_READY.write_text(LOCK_SHA256 + '\n', encoding='utf-8')
    print({'isolated_environment': str(ISOLATED_ENV), 'reused': False, 'isolated_python': isolated_version, 'kernel_python': platform.python_version(), 'locked_packages': LOCKED_PACKAGES, 'setup_seconds': round(time.perf_counter() - setup_started)})

# The worker runs in the isolated environment. It executes each routed cell in one persistent namespace and sends
# back printed text, displayed objects and matplotlib figures, so every cell behaves as it would in the kernel.
_WORKER_SOURCE = r"""
import ast, base64, builtins, io, linecache, os, signal, sys, traceback, types
from multiprocessing.connection import Connection

_send = Connection(int(sys.argv[1]), readable=False)
_recv = Connection(int(sys.argv[2]), writable=False)


class _Stream(io.TextIOBase):
    def __init__(self, name):
        self._name = name

    @property
    def encoding(self):
        return "utf-8"

    def writable(self):
        return True

    def isatty(self):
        return False

    def write(self, text):
        if text:
            _send.send(("stream", self._name, str(text)))
        return len(text)


sys.stdout, sys.stderr = _Stream("stdout"), _Stream("stderr")


def _figure_bundle(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    return {"image/png": base64.b64encode(buffer.getvalue()).decode("ascii"), "text/plain": repr(fig)}


def _flush_figures():
    plt = sys.modules.get("matplotlib.pyplot")
    if plt is None:
        return
    for number in plt.get_fignums():
        _send.send(("display", _figure_bundle(plt.figure(number))))
    plt.close("all")


def _mimebundle(obj):
    if hasattr(obj, "savefig"):
        return _figure_bundle(obj)
    data = {"text/plain": repr(obj)}
    for method, mime in (("_repr_html_", "text/html"), ("_repr_markdown_", "text/markdown"), ("_repr_png_", "image/png")):
        render = getattr(obj, method, None)
        if callable(render):
            try:
                value = render()
            except Exception:
                value = None
            if isinstance(value, bytes):
                value = base64.b64encode(value).decode("ascii")
            if value is not None:
                data[mime] = value
    return data


def display(*objects, **kwargs):
    for obj in objects:
        _send.send(("display", _mimebundle(obj)))


try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot

    matplotlib.pyplot.show = lambda *args, **kwargs: _flush_figures()
except ImportError:
    pass

if os.environ.get("DIMER_KERNEL_IS_COLAB") == "1":
    # google.colab only exists in the kernel; forward the BYOD upload dialog to it.
    def _upload():
        _send.send(("upload",))
        reply = _recv.recv()
        if reply[1] is None:
            raise RuntimeError("The notebook kernel could not open the upload dialog.")
        return reply[1]

    import importlib.machinery

    def _stub(name, package):
        # A spec on every stub: importlib.util.find_spec("google.colab") (accelerate does this) raises on a None __spec__.
        module = types.ModuleType(name)
        module.__spec__ = importlib.machinery.ModuleSpec(name, None, is_package=package)
        if package:
            module.__path__ = []
        return module

    try:
        import google
    except ImportError:
        google = _stub("google", True)
        sys.modules["google"] = google
    _colab = _stub("google.colab", True)
    _files = _stub("google.colab.files", False)
    _files.upload = _upload
    _colab.files = _files
    google.colab = _colab
    sys.modules["google.colab"] = _colab
    sys.modules["google.colab.files"] = _files

_main = types.ModuleType("__main__")
_main.__dict__.update(__builtins__=builtins, display=display)
sys.modules["__main__"] = _main
_count = 0
while True:
    # An interrupt only lands inside a running cell; between cells it is ignored.
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    try:
        message = _recv.recv()
    except EOFError:
        break
    if message[0] != "run":
        continue
    _count += 1
    filename = f"<isolated cell {_count}>"
    source = message[1]
    linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
    try:
        signal.signal(signal.SIGINT, signal.default_int_handler)
        tree = ast.parse(source, filename)
        tail = ast.Expression(tree.body.pop().value) if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        exec(compile(tree, filename, "exec"), _main.__dict__)
        if tail is not None:
            value = eval(compile(tail, filename, "eval"), _main.__dict__)
            if value is not None:
                display(value)
        _flush_figures()
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        _send.send(("done",))
    except BaseException as exc:
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        frames = exc.__traceback__.tb_next if exc.__traceback__ is not None else None
        _send.send(("error", "".join(traceback.format_exception(type(exc), exc, frames)), f"{type(exc).__name__}: {exc}"))
"""


class IsolatedCellError(RuntimeError):
    """A routed cell raised inside the isolated environment; its traceback is printed above."""


class IsolatedRuntime:
    """One persistent worker process in the isolated environment, fed one cell at a time."""

    def __init__(self, python, display=None):
        self.python = str(python)
        to_kernel_r, to_kernel_w = os.pipe()
        to_worker_r, to_worker_w = os.pipe()
        env = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1", DIMER_NOTEBOOK_CI_PREINSTALLED="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            env.pop(name, None)
        env["DIMER_KERNEL_IS_COLAB"] = "1" if "google.colab" in sys.modules else "0"
        self.proc = subprocess.Popen(
            [str(python), "-c", _WORKER_SOURCE, str(to_kernel_w), str(to_worker_r)],
            pass_fds=(to_kernel_w, to_worker_r),
            env=env,
            start_new_session=True,  # interrupts reach the worker only through run(), exactly once
        )
        os.close(to_kernel_w)
        os.close(to_worker_r)
        self._recv = Connection(to_kernel_r, writable=False)
        self._send = Connection(to_worker_w, readable=False)
        if display is None:
            from IPython.display import display
        self._display = display

    def alive(self):
        return self.proc.poll() is None

    def _exited(self):
        return RuntimeError(
            f"The isolated environment's Python process exited (code {self.proc.wait()}); a crash of this kind is "
            "usually running out of memory. Restart the session and choose Run all again."
        )

    def run(self, source):
        try:
            self._send.send(("run", source))
        except OSError:
            raise self._exited() from None
        while True:
            try:
                message = self._recv.recv()
            except EOFError:
                raise self._exited() from None
            except KeyboardInterrupt:
                self.proc.send_signal(signal.SIGINT)
                continue
            kind = message[0]
            if kind == "stream":
                (sys.stdout if message[1] == "stdout" else sys.stderr).write(message[2])
            elif kind == "display":
                self._display(message[1], raw=True)
            elif kind == "upload":
                self._send.send(("upload", self._colab_upload()))
            elif kind == "error":
                sys.stderr.write(message[1])
                raise IsolatedCellError(message[2]) from None
            elif kind == "done":
                return

    @staticmethod
    def _colab_upload():
        try:
            from google.colab import files
        except ImportError:
            return None
        return files.upload()

    def close(self):
        self._send.close()
        self.proc.wait(timeout=30)


def _is_user_cell():
    # ipykernel transforms a cell before executing it; its caller knows whether this is a silent frontend request.
    frame = sys._getframe(2)
    while frame is not None:
        local = frame.f_locals
        if "silent" in local and "store_history" in local:
            return bool(local["store_history"]) and not bool(local["silent"])
        frame = frame.f_back
    return True


def _route_to_isolated_runtime(lines):
    source = "".join(lines)
    if not source.strip() or "# dimer: kernel cell" in source or not _is_user_cell():
        return lines
    return [f"_DIMER_ISOLATED_RUNTIME.run({source!r})\n"]

if SKIP_INSTALL:
    print('Routing disabled: the notebook runs in this kernel.')
else:
    from IPython import get_ipython as _kernel_shell

    _ip = _kernel_shell()
    _ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup if getattr(t, '__name__', '') != '_route_to_isolated_runtime']
    # Idempotent: a live worker on this environment is kept, with every variable the later cells created, so
    # re-running this cell alone never strands the cells after it. A dead or different worker is replaced.
    _previous = globals().get('_DIMER_ISOLATED_RUNTIME')
    _reuse = _previous is not None and getattr(_previous, 'python', None) == str(ISOLATED_PYTHON) and _previous.proc.poll() is None
    if _reuse:
        _DIMER_ISOLATED_RUNTIME = _previous
    else:
        if _previous is not None and _previous.proc.poll() is None:
            _previous.close()
        _DIMER_ISOLATED_RUNTIME = IsolatedRuntime(ISOLATED_PYTHON)
    _ip.input_transformers_cleanup.append(_route_to_isolated_runtime)
    print({'routed_to': str(ISOLATED_PYTHON), 'worker_pid': _DIMER_ISOLATED_RUNTIME.proc.pid, 'worker_reused': _reuse})

{'isolated_environment': '/content/dimer_isolated_env_cbaf70c710fc', 'reused': False, 'isolated_python': '3.12.12', 'kernel_python': '3.13.15', 'locked_packages': 56, 'setup_seconds': 56}
{'routed_to': '/content/dimer_isolated_env_cbaf70c710fc/bin/python', 'worker_pid': 3869, 'worker_reused': False}


### Record the runtime

This is the first cell that runs in the isolated environment. Nothing is installed here. It records the notebook's source revision and the versions actually imported. Look for a dictionary reporting the notebook's source revision, the isolated Python (3.12.12), `torch`, `transformers`, `safetensors` versions, and whether CUDA is available.

In [ ]:
# @title Infrastructure: record the runtime
import importlib
import os
import platform
import sys

NOTEBOOK_SOURCE = {
    'repository': 'molformer-chemistry-pipeline',
    'repository_revision': '6cde8bd0b92f5c7adad35cfb1632df444e74b2cd',
    'embedded_module': 'src/molformer_chemistry_pipeline/pipeline.py',
    'embedded_modules': ['src/molformer_chemistry_pipeline/pipeline.py', 'src/molformer_chemistry_pipeline/samples.py', 'src/molformer_chemistry_pipeline/metrics.py'],
    'module_sha256': '1783c35ea4be6e6c9c614324c2cbd9c99a376a67182a137370f352362f8da134',
    'generator': 'build_notebook.py/2.2',
    'notebook_spec': '2.2',
}
import torch, transformers, safetensors
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'executable': sys.executable, 'torch': torch.__version__, 'transformers': transformers.__version__, 'safetensors': safetensors.__version__, 'cuda': torch.cuda.is_available()})

{'notebook_source': {'repository': 'molformer-chemistry-pipeline', 'repository_revision': '6cde8bd0b92f5c7adad35cfb1632df444e74b2cd', 'embedded_module': 'src/molformer_chemistry_pipeline/pipeline.py', 'embedded_modules': ['src/molformer_chemistry_pipeline/pipeline.py', 'src/molformer_chemistry_pipeline/samples.py', 'src/molformer_chemistry_pipeline/metrics.py'], 'module_sha256': '1783c35ea4be6e6c9c614324c2cbd9c99a376a67182a137370f352362f8da134', 'generator': 'build_notebook.py/2.2', 'notebook_spec': '2.2'}, 'python': '3.12.12', 'executable': '/content/dimer_isolated_env_cbaf70c710fc/bin/python', 'torch': '2.14.0+cu130', 'transformers': '5.17.0', 'safetensors': '0.8.0', 'cuda': True}


## 2. Pipeline code (carried verbatim from `src/molformer_chemistry_pipeline/` @ `6cde8bd0b92f`)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/molformer_chemistry_pipeline/pipeline.py`

In [ ]:
"""MoLFormer-XL (`ibm-research/MoLFormer-XL-both-10pct`) DIMER pipeline: verified snapshot, SMILES
embeddings, and bounded molecular-property classification fine-tuning with a portable adapter.

Two things about this checkpoint shape the module and are stated rather than hidden:

* **It requires remote code.** `config.json` declares `model_type: "molformer"`, for which the
  installed Transformers has no native implementation, and the architecture is a linear-attention
  transformer that the generic classes do not implement. The loader therefore passes
  `trust_remote_code=True` — but only after `verify_snapshot` has checked the SHA-256 of the two
  Python files it will import, which are manifest entries for exactly that reason.
* **Its pinned code needs a recent Transformers.** The upstream revision pinned here calls
  `transformers.masking_utils.create_bidirectional_mask(inputs_embeds=...)`, which exists only in
  Transformers 5.5.0 and later, so this package pins `transformers==5.17.0` rather than the fleet's
  4.57.6. See `docs/WEIGHTS.md`.
* **Its attention is stochastic unless told otherwise.** MoLFormer uses linear attention with
  random feature maps. The upstream config ships `deterministic_eval: false`, which redraws those
  features on *every* forward pass, so two identical calls return different embeddings (measured:
  ~1e-3 on a pooled vector). This package loads with `deterministic_eval=True`, which keeps the
  feature weights the checkpoint itself stores, and exports those buffers with any adapter because
  they are serving state, not incidental.

Everything model-related is imported lazily so that snapshot verification and input validation run
(and can refuse) before `torch` or `transformers` are imported (fleet RTM-001).
"""

from __future__ import annotations

import hashlib
import json
import warnings
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

MODEL_ID = "ibm-research/MoLFormer-XL-both-10pct"
MODEL_REVISION = "361063d0ad524ef77cf39b08469f6be770dc550f"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "molformer-xl-both-10pct"
ARTIFACT_FORMAT = "org.valcorza.molformer-chemistry.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
# The two files the loader executes. They are manifest entries, so their bytes are digest-verified
# before `trust_remote_code=True` imports them (MODEL_ASSET_SPEC RC3).
REMOTE_CODE_FILES = ("configuration_molformer.py", "modeling_molformer.py")
# Linear-attention random-feature buffers. They live in the checkpoint, they change during training,
# and inference depends on them, so an exported adapter carries them (ART3).
FEATURE_MAP_SUFFIX = ".feature_map.weight"
# Upstream defaults to redrawing the random features on every forward pass (`deterministic_eval:
# false` in config.json). DIMER needs a reproducible answer to the same question, so this package
# overrides that to True, which uses the feature weights stored in the pinned checkpoint.
DETERMINISTIC_EVAL = True

# Ceilings. config.json declares max_position_embeddings = 202, and the tokenizer wraps every SMILES
# in <bos> ... <eos>, so 200 SMILES tokens is the most a molecule can carry. Longer input is refused
# rather than truncated: a truncated SMILES is a different molecule, not a shorter one.
MAX_POSITION_EMBEDDINGS = 202
MAX_TOKENS = MAX_POSITION_EMBEDDINGS - 2
MAX_MOLECULES_PER_CALL = 64
HIDDEN_SIZE = 768  # config.json hidden_size; the width of every `embed` row
VOCAB_SIZE = 2362  # config.json vocab_size; equals the tokenizer's vocabulary
# A syntactic character set for SMILES, not a chemical validity check: this package has no cheminformatics
# toolkit, so it checks characters, bracket balance and ring-digit pairing and says so. Use RDKit to
# establish that a string is a real molecule.
SMILES_CHARS = frozenset(
    "BCNOPSFIHbcnopsfi"  # organic subset, aromatic lowercase, explicit H in brackets
    "lr"  # the second letters of Cl and Br
    "eaigAKLMTZ"  # letters that appear inside bracket atoms (Se, Na, Si, Mg, ...)
    "0123456789"  # ring-closure digits
    "()[]"  # branches and bracket atoms
    "=#$:/\\"  # bond symbols
    "+-@"  # charges and stereochemistry
    "%."  # two-digit ring closures and disconnected components
)


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    listed = {entry["path"] for entry in manifest["files"]}
    missing_code = [name for name in REMOTE_CODE_FILES if name not in listed]
    if missing_code:
        raise ValueError(
            f"manifest does not list the remote-code files {missing_code}; refusing to proceed, because "
            "the loader executes them and their digests must be verified first"
        )
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = hashlib.sha256()
        with open(file_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                digest.update(chunk)
        if digest.hexdigest() != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest.hexdigest()} != manifest {entry['sha256']}")
    return manifest


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the snapshot against its DIMER manifest, including the two executable Python files."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _verify_manifest(root, MODEL_ID, MODEL_REVISION)


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest entries that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights and the upstream Python). `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


INPUT_SCHEMA: dict[str, Any] = {
    "input": "1..MAX_MOLECULES_PER_CALL molecules as SMILES strings",
    "molecules": [1, MAX_MOLECULES_PER_CALL],
    "tokens_per_molecule": [1, MAX_TOKENS],
    "validation": (
        "syntactic only: character set, balanced parentheses and brackets, paired ring-closure digits, "
        "and the tokenized length ceiling. This package ships no cheminformatics toolkit and does not "
        "check valence, aromaticity or chemical plausibility -- use RDKit for that"
    ),
    "preprocessing": (
        "the pinned tokenizer splits SMILES into atom and symbol tokens (two-letter atoms such as Cl and "
        "Br and bracket atoms such as [C@H] stay single tokens), wraps them in <bos> ... <eos>, and pads "
        "to the longest molecule in the batch; embeddings are the mean of the last hidden state over "
        "non-padding tokens"
    ),
}


def check_smiles_syntax(smiles: str) -> list[str]:
    """Return a list of syntactic findings for one SMILES string; empty means it passed the checks.

    This is a string-level check, not a chemistry check: it catches typos and truncation, not
    impossible molecules.
    """
    findings: list[str] = []
    bad = sorted({ch for ch in smiles if ch not in SMILES_CHARS})
    if bad:
        findings.append(f"characters outside the accepted SMILES set: {bad!r}")
    for opener, closer, label in (("(", ")", "parenthesis"), ("[", "]", "bracket")):
        depth = 0
        for ch in smiles:
            depth += ch == opener
            depth -= ch == closer
            if depth < 0:
                findings.append(f"closing {label} before an opening one")
                break
        if depth > 0:
            findings.append(f"{depth} unclosed {label}(s)")
    digits: dict[str, int] = {}
    in_bracket = False
    for ch in smiles:
        if ch == "[":
            in_bracket = True
        elif ch == "]":
            in_bracket = False
        elif ch.isdigit() and not in_bracket:
            digits[ch] = digits.get(ch, 0) + 1
    odd = sorted(d for d, count in digits.items() if count % 2)
    if odd:
        findings.append(f"unpaired ring-closure digit(s): {odd}")
    return findings


def _check_molecules(molecules: Any, names: Any = None) -> tuple[list[str], list[str]]:
    """Raise TypeError/ValueError naming the first violated rule; return (smiles, ids).

    ``embed``, ``classify`` and ``validate_inputs`` all route through this function so their
    acceptance criteria cannot diverge.
    """
    if isinstance(molecules, str | bytes) or not isinstance(molecules, Sequence):
        raise TypeError("molecules must be a list of SMILES strings")
    if not 1 <= len(molecules) <= MAX_MOLECULES_PER_CALL:
        raise ValueError(f"molecules must hold 1..{MAX_MOLECULES_PER_CALL} items, got {len(molecules)}")
    checked: list[str] = []
    for i, smiles in enumerate(molecules):
        if not isinstance(smiles, str):
            raise TypeError(f"molecules[{i}] must be str, got {type(smiles).__name__}")
        if not smiles.strip():
            raise ValueError(f"molecules[{i}] is empty")
        if smiles != smiles.strip():
            raise ValueError(f"molecules[{i}] has leading or trailing whitespace; strip it first")
        findings = check_smiles_syntax(smiles)
        if findings:
            raise ValueError(
                f"molecules[{i}] ({smiles!r}) failed SMILES syntax checks: {'; '.join(findings)}"
            )
        checked.append(smiles)
    if names is None:
        ids = [f"mol-{i}" for i in range(len(checked))]
    else:
        if isinstance(names, str | bytes) or not isinstance(names, Sequence) or len(names) != len(checked):
            raise ValueError("names must be a list with exactly one id per molecule")
        ids = [str(n) for n in names]
        if len(set(ids)) != len(ids):
            raise ValueError("names must be unique")
    return checked, ids


def _softmax(logits: Sequence[float]) -> list[float]:
    import math

    top = max(logits)
    exps = [math.exp(v - top) for v in logits]
    total = sum(exps)
    return [v / total for v in exps]


@dataclass
class MolformerPipeline:
    """MoLFormer-XL pipeline: `embed` always; `classify` after `adapt` or `from_artifact`."""

    _embedder: Callable[[list[str]], list[list[float]]]
    device: str
    load_warnings: list[str] = field(default_factory=list)
    classes: list[str] = field(default_factory=list)
    _classifier: Callable[[list[str]], list[list[float]]] | None = None
    _token_counter: Callable[[str], int] | None = None
    model: Any = None
    tokenizer: Any = None
    classifier_model: Any = None
    weights_dir: Path | None = None
    adaptation: dict[str, Any] = field(default_factory=dict)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> MolformerPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if not (root / MANIFEST_NAME).is_file():
            raise FileNotFoundError(f"no snapshot manifest at {root} and allow_download={allow_download}")
        # Stage and verify — including the two Python files — before importing model libraries and
        # before any remote code is executed (RTM-001, MODEL_ASSET_SPEC RC3).
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        import torch
        from transformers import AutoModel, AutoTokenizer

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            # The tokenizer is a native PreTrainedTokenizerFast: no remote code for this half.
            tokenizer = AutoTokenizer.from_pretrained(
                str(root), local_files_only=True, trust_remote_code=False
            )
            model = AutoModel.from_pretrained(
                str(root),
                local_files_only=True,
                trust_remote_code=True,
                deterministic_eval=DETERMINISTIC_EVAL,
            )
        model = model.to(resolved_device).eval()
        messages = [f"{w.category.__name__}: {w.message}" for w in caught]
        pipe = cls(cls._make_embedder(model, tokenizer, resolved_device), resolved_device, messages)
        pipe.model, pipe.tokenizer, pipe.weights_dir = model, tokenizer, root
        pipe._token_counter = lambda smiles: len(tokenizer(smiles)["input_ids"])
        return pipe

    # -- backends ---------------------------------------------------------------------------------

    @staticmethod
    def _encode(tokenizer: Any, molecules: list[str], device: str) -> dict[str, Any]:
        batch = tokenizer(molecules, return_tensors="pt", padding=True)
        return {k: v.to(device) for k, v in batch.items() if k in ("input_ids", "attention_mask")}

    @classmethod
    def _make_embedder(
        cls, model: Any, tokenizer: Any, device: str
    ) -> Callable[[list[str]], list[list[float]]]:
        import torch

        def embedder(molecules: list[str]) -> list[list[float]]:
            batch = cls._encode(tokenizer, molecules, device)
            # no_grad, not inference_mode: tensors produced here must stay usable by a later
            # training epoch that shares this module.
            with torch.no_grad():
                hidden = model(**batch).last_hidden_state
            mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
            return pooled.float().cpu().tolist()

        return embedder

    @classmethod
    def _make_classifier(
        cls, model: Any, tokenizer: Any, device: str
    ) -> Callable[[list[str]], list[list[float]]]:
        import torch

        def classifier(molecules: list[str]) -> list[list[float]]:
            batch = cls._encode(tokenizer, molecules, device)
            with torch.no_grad():
                logits = model(**batch).logits
            return logits.float().cpu().tolist()

        return classifier

    def token_count(self, smiles: str) -> int:
        """Tokenized length including `<bos>`/`<eos>`; the ceiling applies to this, not to characters."""
        if self._token_counter is None:
            raise RuntimeError("token_count requires a pipeline built by from_pretrained")
        return self._token_counter(smiles)

    def _check_lengths(self, molecules: Sequence[str]) -> list[int]:
        counts = []
        for i, smiles in enumerate(molecules):
            n_tokens = self.token_count(smiles) - 2  # exclude <bos>/<eos>
            if n_tokens > MAX_TOKENS:
                raise ValueError(
                    f"molecules[{i}] tokenizes to {n_tokens} tokens; the ceiling is {MAX_TOKENS} "
                    f"(max_position_embeddings {MAX_POSITION_EMBEDDINGS} minus <bos> and <eos>). "
                    "A truncated SMILES is a different molecule, so this is refused rather than cut."
                )
            counts.append(n_tokens)
        return counts

    # -- public stages ----------------------------------------------------------------------------

    def embed(self, molecules: Sequence[str], *, names: Sequence[str] | None = None) -> dict[str, Any]:
        """Mean-pooled last-hidden-state representation per molecule (HIDDEN_SIZE floats each)."""
        checked, ids = _check_molecules(molecules, names)
        token_counts = self._check_lengths(checked)
        vectors = self._embedder(checked)
        if len(vectors) != len(checked) or any(len(v) != HIDDEN_SIZE for v in vectors):
            raise RuntimeError("backend returned embeddings of the wrong shape")
        return {
            "ids": ids,
            "embeddings": [[float(x) for x in v] for v in vectors],
            "dimension": HIDDEN_SIZE,
            "pooling": "mean of the last hidden state over non-padding tokens",
            "unit": "one vector per molecule; representations, not property predictions",
            "tokens": token_counts,
            "n_molecules": len(checked),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def classify(self, molecules: Sequence[str], *, names: Sequence[str] | None = None) -> dict[str, Any]:
        """Class scores and argmax label per molecule; requires a prior `adapt` or `from_artifact`."""
        if self._classifier is None or not self.classes:
            raise RuntimeError(
                "classify requires an adapted head: call adapt(...) or load from_artifact(...) first"
            )
        checked, ids = _check_molecules(molecules, names)
        token_counts = self._check_lengths(checked)
        logits = self._classifier(checked)
        predictions = []
        for mid, smiles, n_tokens, row in zip(ids, checked, token_counts, logits, strict=True):
            if len(row) != len(self.classes):
                raise RuntimeError("backend returned a logits row that does not match the class list")
            scores = _softmax(row)
            best = max(range(len(scores)), key=scores.__getitem__)
            predictions.append(
                {
                    "id": mid,
                    "smiles": smiles,
                    "tokens": n_tokens,
                    "label": self.classes[best],
                    "score": scores[best],
                    "scores": dict(zip(self.classes, scores, strict=True)),
                }
            )
        return {
            "predictions": predictions,
            "classes": list(self.classes),
            "decision_rule": (
                "argmax over softmax(logits); scores are softmax outputs, not calibrated probabilities"
            ),
            "n_molecules": len(checked),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "adaptation": dict(self.adaptation),
        }

    def evaluate(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Held-out property-classification metrics (see metrics.classification_metrics)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import classification_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        validate_dataset(records, classes=self.classes)
        predicted: list[str] = []
        scores: list[list[float]] = []
        for start in range(0, len(records), MAX_MOLECULES_PER_CALL):
            chunk = records[start : start + MAX_MOLECULES_PER_CALL]
            result = self.classify([r["smiles"] for r in chunk], names=[r["id"] for r in chunk])
            for p in result["predictions"]:
                predicted.append(p["label"])
                scores.append([p["scores"][c] for c in self.classes])
        return classification_metrics([r["label"] for r in records], predicted, scores, self.classes)

    def adapt(
        self,
        train_records: Sequence[Mapping[str, Any]],
        val_records: Sequence[Mapping[str, Any]] | None = None,
        *,
        classes: Sequence[str] | None = None,
        epochs: int = 4,
        learning_rate: float = 1e-4,
        batch_size: int = 8,
        trainable_layers: int = 2,
        weight_decay: float = 0.01,
        seed: int = 42,
    ) -> dict[str, Any]:
        """Bounded gradient fine-tuning of a property-classification head on the verified base.

        Builds `MolformerForSequenceClassification` from the pinned checkpoint (the head is newly
        initialised), freezes every parameter except the head and the last `trainable_layers`
        encoder layers, and runs AdamW for `epochs` passes. Validation records are monitored per
        epoch only; the final epoch's weights are kept (no selection).
        """
        if self.model is None or self.tokenizer is None or self.weights_dir is None:
            raise RuntimeError("adapt requires a pipeline built by from_pretrained (no loaded base model)")
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if not 1 <= int(epochs) <= 50:
            raise ValueError("epochs must be in 1..50 (tutorial-scale adaptation)")
        if not 1 <= int(batch_size) <= MAX_MOLECULES_PER_CALL:
            raise ValueError(f"batch_size must be in 1..{MAX_MOLECULES_PER_CALL}")
        if not 0 <= int(trainable_layers) <= 12:
            raise ValueError("trainable_layers must be in 0..12 (the checkpoint has 12 encoder layers)")
        train_manifest = validate_dataset(train_records, classes=classes)
        class_list = list(train_manifest["classes"])
        if val_records is not None:
            validate_dataset(val_records, classes=class_list)

        import random

        import torch
        from transformers import AutoModelForSequenceClassification

        random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        clf = AutoModelForSequenceClassification.from_pretrained(
            str(self.weights_dir),
            local_files_only=True,
            trust_remote_code=True,
            num_labels=len(class_list),
            deterministic_eval=DETERMINISTIC_EVAL,
        ).to(self.device)
        for p in clf.parameters():
            p.requires_grad = False
        layers = clf.molformer.encoder.layer
        for layer in layers[len(layers) - int(trainable_layers) :] if trainable_layers else []:
            for p in layer.parameters():
                p.requires_grad = True
        for p in clf.classifier.parameters():
            p.requires_grad = True
        trainable = [n for n, p in clf.named_parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in clf.parameters() if p.requires_grad)
        n_total = sum(p.numel() for p in clf.parameters())
        optimizer = torch.optim.AdamW(
            [p for p in clf.parameters() if p.requires_grad], lr=learning_rate, weight_decay=weight_decay
        )
        label_index = {c: i for i, c in enumerate(class_list)}
        examples = [(r["smiles"], label_index[r["label"]]) for r in train_records]
        self.classes = class_list
        self.classifier_model = clf
        self._classifier = self._make_classifier(clf, self.tokenizer, self.device)

        history: list[dict[str, Any]] = []
        for epoch in range(1, int(epochs) + 1):
            clf.train()
            order = list(range(len(examples)))
            random.shuffle(order)
            total_loss, n_batches = 0.0, 0
            for start in range(0, len(order), int(batch_size)):
                rows = [examples[i] for i in order[start : start + int(batch_size)]]
                batch = self._encode(self.tokenizer, [s for s, _ in rows], self.device)
                labels = torch.tensor([y for _, y in rows], device=self.device)
                optimizer.zero_grad()
                out = clf(**batch, labels=labels)
                out.loss.backward()
                optimizer.step()
                total_loss += float(out.loss.item())
                n_batches += 1
            clf.eval()
            entry: dict[str, Any] = {
                "epoch": epoch,
                "train_loss": round(total_loss / max(1, n_batches), 6),
                "n_batches": n_batches,
            }
            if val_records:
                val = self.evaluate(val_records)
                entry["val_accuracy"] = val["accuracy"]
                entry["val_macro_f1"] = val["macro_f1"]
            history.append(entry)
        clf.eval()
        self.adaptation = {
            "method": "gradient fine-tuning (AdamW) of the classification head"
            + (f" and the last {int(trainable_layers)} encoder layer(s)" if trainable_layers else ""),
            "classes": class_list,
            "epochs": int(epochs),
            "learning_rate": float(learning_rate),
            "batch_size": int(batch_size),
            "weight_decay": float(weight_decay),
            "trainable_layers": int(trainable_layers),
            "seed": int(seed),
            "precision": "float32",
            "trainable_parameters": int(n_trainable),
            "total_parameters": int(n_total),
            "trainable_parameter_names": trainable,
            "train_records": len(train_records),
            "val_records": len(val_records) if val_records else 0,
            "selection": "final epoch kept; validation metrics are monitoring only",
            "history": history,
        }
        return dict(self.adaptation)

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Export the trainable tensors as safetensors plus a JSON manifest binding them to the base."""
        if self.classifier_model is None or not self.classes:
            raise RuntimeError("save_artifact requires an adapted head (call adapt first)")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adaptation.get("trainable_parameter_names", []))
        state = self.classifier_model.state_dict()
        # Trained parameters plus the linear-attention feature buffers: training redraws those
        # buffers, and inference depends on them, so an adapter without them does not reproduce
        # the adapted model (ART3).
        serving_state = sorted(k for k in state if k.endswith(FEATURE_MAP_SUFFIX))
        tensors = {
            k: v.detach().cpu().contiguous() for k, v in state.items() if k in names or k in serving_state
        }
        if not tensors:
            raise RuntimeError("no trainable tensors recorded; nothing to export")
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path))
        digest = hashlib.sha256(weights_path.read_bytes()).hexdigest()
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {"model_id": MODEL_ID, "model_revision": MODEL_REVISION, "license": MODEL_LICENSE},
            "requires_remote_code": True,
            "classes": list(self.classes),
            "files": [
                {"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": digest}
            ],
            "tensors": sorted(tensors),
            "serving_state_tensors": serving_state,
            "deterministic_eval": DETERMINISTIC_EVAL,
            "adaptation": {k: v for k, v in self.adaptation.items() if k != "trainable_parameter_names"},
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Rebuild the classification head from an exported artifact (manifest verified before loading)."""
        if self.model is None or self.weights_dir is None:
            raise RuntimeError("load_artifact requires a pipeline built by from_pretrained")
        art = Path(artifact_dir)
        manifest_path = art / ARTIFACT_MANIFEST_NAME
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest not found: {manifest_path}")
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        base = manifest.get("base_model", {})
        if (base.get("model_id"), base.get("model_revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError(f"artifact was trained on {base}, this package pins {MODEL_ID}@{MODEL_REVISION}")
        classes = [str(c) for c in manifest.get("classes", [])]
        if len(classes) < 2 or len(set(classes)) != len(classes):
            raise ValueError("artifact manifest must list at least two unique classes")
        for entry in manifest["files"]:
            fp = art / entry["path"]
            if not fp.is_file():
                raise FileNotFoundError(f"artifact file missing: {fp}")
            if fp.stat().st_size != entry["bytes"]:
                raise ValueError(f"{entry['path']}: size {fp.stat().st_size} != manifest {entry['bytes']}")
            if hashlib.sha256(fp.read_bytes()).hexdigest() != entry["sha256"]:
                raise ValueError(f"{entry['path']}: sha256 mismatch against the artifact manifest")
        from safetensors.torch import load_file
        from transformers import AutoModelForSequenceClassification

        clf = AutoModelForSequenceClassification.from_pretrained(
            str(self.weights_dir),
            local_files_only=True,
            trust_remote_code=True,
            num_labels=len(classes),
            deterministic_eval=DETERMINISTIC_EVAL,
        )
        tensors = load_file(str(art / ARTIFACT_WEIGHTS_NAME))
        if set(tensors) != set(manifest.get("tensors", [])):
            raise ValueError("artifact tensors do not match the names listed in its manifest")
        _missing, unexpected = clf.load_state_dict(tensors, strict=False)
        if unexpected:
            raise ValueError(
                f"artifact carries tensors the base architecture does not have: {sorted(unexpected)[:5]}"
            )
        clf = clf.to(self.device).eval()
        self.classes = classes
        self.classifier_model = clf
        self._classifier = self._make_classifier(clf, self.tokenizer, self.device)
        self.adaptation = {**manifest.get("adaptation", {}), "loaded_from_artifact": str(art)}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> MolformerPipeline:
        """Verified base snapshot + exported adapter, ready for `classify`."""
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe


def validate_inputs(
    molecules: Sequence[str],
    *,
    names: Sequence[str] | None = None,
    token_counter: Callable[[str], int] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, verdict).

    `token_counter` is the pipeline's `token_count`; without it the manifest reports character
    lengths and says that the token ceiling was not checked, rather than guessing at it.
    """
    checked, ids = _check_molecules(molecules, names)
    rows = []
    for mid, smiles in zip(ids, checked, strict=True):
        row: dict[str, Any] = {"id": mid, "smiles": smiles, "characters": len(smiles)}
        if token_counter is not None:
            n_tokens = token_counter(smiles) - 2
            if n_tokens > MAX_TOKENS:
                raise ValueError(
                    f"{mid} tokenizes to {n_tokens} tokens; the ceiling is {MAX_TOKENS} "
                    f"(max_position_embeddings {MAX_POSITION_EMBEDDINGS} minus <bos> and <eos>)"
                )
            row["tokens"] = n_tokens
        rows.append(row)
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": rows,
        "n_molecules": len(checked),
        "token_ceiling_checked": token_counter is not None,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "requires_remote_code": True,
    }

**Module 2/3:** `src/molformer_chemistry_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Deterministic in-code sample molecules and the labelled-dataset contract for property classification.

The tutorial task is synthetic but chemically literal: each pair of molecules is a pair of
**constitutional isomers** — the same molecular formula, differing only in connectivity. One member
is an unbranched primary alcohol (`CCCCCCCO`); the other carries the same atoms with one methyl
branch moved onto the chain (`CCCC(C)CCO`). Molecular formula therefore carries no signal by
construction, so a composition baseline sits at chance and any separation the model achieves comes
from reading structure. This is sanity evidence for the adaptation contract, not a chemistry
benchmark, and the classes are a generator rule rather than a measured property (NOTEBOOK_SPEC 2.0
DAT8).

One honest caveat, stated here and in the tutorial: a branched SMILES contains `(` and `)`, so a
*character-level* baseline that counted punctuation could separate these classes trivially. The
baseline this package ships counts heavy atoms — the molecular formula — because that is the
chemically meaningful confounder to rule out.
"""

from __future__ import annotations

import csv
import hashlib
import json
import random
import re
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_TOKENS, check_smiles_syntax` removed — names are kernel globals defined by the carried modules

DATASET_REPRESENTATION = "io.github.kurtvalcorza.dataset.chemistry.smiles-labels.v1"
SAMPLE_CLASSES: tuple[str, ...] = ("branched", "linear")
SAMPLE_SEED = 20260918
SAMPLE_SIZE = 64  # 32 pairs
MIN_CARBONS = 6
MAX_CARBONS = 14
# Terminal groups, so that pairs differ from one another rather than only in chain length. Both
# members of a pair carry the same terminus, so the terminus can never separate the two classes.
TERMINI: tuple[str, ...] = ("O", "N", "S", "F", "Cl", "Br", "I")
MIN_RECORDS = 8
MAX_RECORDS = 5_000
MAX_CLASSES = 20
MIN_RECORDS_PER_CLASS = 3
MAX_ID_CHARS = 64
MAX_LABEL_CHARS = 64
REQUIRED_COLUMNS = ("id", "smiles", "label")
_ATOM = re.compile(r"Cl|Br|[BCNOPSFI]|\[[^\]]*\]|[bcnops]")


def molecular_formula(smiles: str) -> dict[str, int]:
    """Heavy-atom counts for the organic subset this sample uses (no implicit hydrogens).

    This is a string-level count, not a cheminformatics parse: it recognises the two-letter atoms
    `Cl` and `Br`, single-letter organic-subset atoms, aromatic lowercase atoms, and bracket atoms
    as one atom each. It exists to give the tutorial a composition baseline, not to replace RDKit.
    """
    counts: dict[str, int] = {}
    for token in _ATOM.findall(smiles):
        if token.startswith("["):
            inner = token[1:-1]
            match = re.search(r"[A-Z][a-z]?|[bcnops]", inner)
            symbol = match.group(0) if match else inner
        else:
            symbol = token
        symbol = symbol.capitalize() if len(symbol) > 1 else symbol.upper()
        counts[symbol] = counts.get(symbol, 0) + 1
    return counts


def formula_string(smiles: str) -> str:
    """`molecular_formula` rendered in a stable order, e.g. `C7O1`."""
    counts = molecular_formula(smiles)
    return "".join(f"{symbol}{counts[symbol]}" for symbol in sorted(counts))


def branch_positions(n_carbons: int) -> list[int]:
    """Positions at which a methyl branch can sit without recreating the linear chain."""
    return list(range(2, n_carbons - 2))


def make_isomer_pair(n_carbons: int, branch_at: int, terminus: str = "O") -> tuple[str, str]:
    """A linear molecule and a branched constitutional isomer of it: same formula, different skeleton.

    `CCCCCCCO` and `CCCC(C)CCO` both have `n_carbons` carbons and one oxygen: the branched member
    spends one of its carbons as a methyl substituent instead of extending the chain. `terminus` is
    the group at the end of the chain and is identical for both members, so it cannot carry the label.
    """
    if not MIN_CARBONS <= n_carbons <= MAX_CARBONS:
        raise ValueError(f"n_carbons must be in {MIN_CARBONS}..{MAX_CARBONS}, got {n_carbons}")
    if branch_at not in branch_positions(n_carbons):
        raise ValueError(f"branch_at must be one of {branch_positions(n_carbons)} for {n_carbons} carbons")
    if terminus not in TERMINI:
        raise ValueError(f"terminus must be one of {list(TERMINI)}, got {terminus!r}")
    linear = "C" * n_carbons + terminus
    chain = n_carbons - 1  # one carbon becomes the branch
    branched = "C" * branch_at + "(C)" + "C" * (chain - branch_at) + terminus
    return linear, branched


def generate_sample_dataset(seed: int = SAMPLE_SEED, size: int = SAMPLE_SIZE) -> list[dict[str, Any]]:
    """`size` labelled molecules (half `linear`, half `branched`), deterministic for a given seed.

    Every branched molecule is a constitutional isomer of its linear partner: identical formula,
    different connectivity.
    """
    if size < 2 or size % 2:
        raise ValueError("size must be an even number >= 2 (one linear molecule per branched isomer)")
    skeletons = [(n, terminus) for terminus in TERMINI for n in range(MIN_CARBONS, MAX_CARBONS + 1)]
    if size // 2 > len(skeletons):
        raise ValueError(
            f"size {size} needs {size // 2} distinct (chain length, terminus) pairs but only "
            f"{len(skeletons)} exist; widen MIN_CARBONS..MAX_CARBONS or TERMINI"
        )
    rng = random.Random(seed)
    rng.shuffle(skeletons)
    records: list[dict[str, Any]] = []
    for i, (n_carbons, terminus) in enumerate(skeletons[: size // 2]):
        branch_at = rng.choice(branch_positions(n_carbons))
        linear, branched = make_isomer_pair(n_carbons, branch_at, terminus)
        records.append({"id": f"lin-{i:03d}", "smiles": linear, "label": "linear"})
        records.append({"id": f"bra-{i:03d}", "smiles": branched, "label": "branched"})
    return records


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """SHA-256 over the canonical (id, smiles, label) rows; recorded in provenance (OUT9)."""
    canon = json.dumps([[r["id"], r["smiles"], r["label"]] for r in records], separators=(",", ":"))
    return hashlib.sha256(canon.encode("utf-8")).hexdigest()


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    classes: Sequence[str] | None = None,
    min_records: int = MIN_RECORDS,
    min_per_class: int = MIN_RECORDS_PER_CLASS,
) -> dict[str, Any]:
    """Check a labelled molecule dataset against the contract; return its manifest.

    Every error names the record and the violated rule (VAL4/DAT19). SMILES are checked
    syntactically only — character set, bracket balance, ring-digit pairing — because this package
    ships no cheminformatics toolkit; the manifest says so rather than implying chemical validation.
    """
    if isinstance(records, str | bytes | Mapping) or not isinstance(records, Sequence):
        raise TypeError("records must be a list of {'id', 'smiles', 'label'} mappings")
    if len(records) < min_records:
        raise ValueError(f"dataset has {len(records)} records; at least {min_records} are required")
    if len(records) > MAX_RECORDS:
        raise ValueError(f"dataset has {len(records)} records; ceiling is {MAX_RECORDS}")
    seen_ids: set[str] = set()
    seen_smiles: dict[str, str] = {}
    counts_by_class: dict[str, int] = {}
    lengths: list[int] = []
    formulas: dict[str, set[str]] = {}
    for i, rec in enumerate(records):
        if not isinstance(rec, Mapping):
            raise TypeError(f"record[{i}] must be a mapping, got {type(rec).__name__}")
        missing = [c for c in REQUIRED_COLUMNS if c not in rec]
        if missing:
            raise ValueError(
                f"record[{i}] is missing required column(s) {missing}; required: {list(REQUIRED_COLUMNS)}"
            )
        rid = str(rec["id"]).strip()
        if not rid or len(rid) > MAX_ID_CHARS:
            raise ValueError(f"record[{i}] id must be 1..{MAX_ID_CHARS} characters")
        if rid in seen_ids:
            raise ValueError(f"record[{i}] duplicates id {rid!r}")
        seen_ids.add(rid)
        smiles = rec["smiles"]
        if not isinstance(smiles, str) or not smiles.strip():
            raise ValueError(f"record[{i}] ({rid}) smiles must be a non-empty string")
        if smiles != smiles.strip():
            raise ValueError(f"record[{i}] ({rid}) smiles has leading or trailing whitespace")
        findings = check_smiles_syntax(smiles)
        if findings:
            raise ValueError(f"record[{i}] ({rid}) failed SMILES syntax checks: {'; '.join(findings)}")
        if smiles in seen_smiles:
            raise ValueError(f"record[{i}] ({rid}) duplicates the SMILES of {seen_smiles[smiles]!r}")
        seen_smiles[smiles] = rid
        lengths.append(len(smiles))
        label = rec["label"]
        if not isinstance(label, str) or not label.strip() or len(label) > MAX_LABEL_CHARS:
            raise ValueError(
                f"record[{i}] ({rid}) label must be a non-empty string of at most {MAX_LABEL_CHARS} chars"
            )
        counts_by_class[label] = counts_by_class.get(label, 0) + 1
        formulas.setdefault(formula_string(smiles), set()).add(label)
    if classes is None:
        class_list = sorted(counts_by_class)
    else:
        class_list = [str(c) for c in classes]
        unknown = sorted(set(counts_by_class) - set(class_list))
        if unknown:
            raise ValueError(f"labels {unknown} are not in the class list {class_list}")
    if len(class_list) < 2:
        raise ValueError(f"classification needs at least 2 classes, found {class_list}")
    if len(class_list) > MAX_CLASSES:
        raise ValueError(f"{len(class_list)} classes exceeds the ceiling of {MAX_CLASSES}")
    thin = [c for c in class_list if counts_by_class.get(c, 0) < min_per_class]
    if thin:
        raise ValueError(f"classes {thin} have fewer than {min_per_class} records each (class coverage rule)")
    shared = sorted(f for f, labels in formulas.items() if len(labels) > 1)
    return {
        "verdict": "accepted",
        "representation": DATASET_REPRESENTATION,
        "validation": "syntactic SMILES checks only; no cheminformatics toolkit is used",
        "n_records": len(records),
        "classes": class_list,
        "class_counts": {c: counts_by_class.get(c, 0) for c in class_list},
        "smiles_characters": {
            "min": min(lengths),
            "max": max(lengths),
            "mean": round(sum(lengths) / len(lengths), 1),
        },
        "distinct_formulas": len(formulas),
        "formulas_shared_across_classes": len(shared),
        "ceilings": {
            "max_tokens": MAX_TOKENS,
            "max_records": MAX_RECORDS,
            "max_classes": MAX_CLASSES,
            "min_records": min_records,
            "min_records_per_class": min_per_class,
        },
        "digest": dataset_digest(records),
        "findings": [],
    }


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.2,
    test_fraction: float = 0.25,
    seed: int = 42,
) -> dict[str, list[dict[str, Any]]]:
    """Stratified random train/validation/test split (assumes independent molecules, SPL3).

    Real chemical datasets are rarely independent: analogues from one series share a scaffold, and a
    random split lets the model memorise it. A scaffold or cluster split is the right tool there;
    the tutorial's molecules are generated independently, which is why a random split is honest here.
    """
    if not (0.0 < val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("val_fraction and test_fraction must be in (0, 1) and sum to less than 1")
    manifest = validate_dataset(records)
    rng = random.Random(seed)
    by_class: dict[str, list[dict[str, Any]]] = {c: [] for c in manifest["classes"]}
    for rec in records:
        by_class[rec["label"]].append(dict(rec))
    out: dict[str, list[dict[str, Any]]] = {"train": [], "validation": [], "test": []}
    for cls in manifest["classes"]:
        rows = by_class[cls]
        rng.shuffle(rows)
        n_val = max(1, round(len(rows) * val_fraction))
        n_test = max(1, round(len(rows) * test_fraction))
        if len(rows) - n_val - n_test < 1:
            raise ValueError(f"class {cls!r} has {len(rows)} records; too few to leave one per split")
        out["validation"].extend(rows[:n_val])
        out["test"].extend(rows[n_val : n_val + n_test])
        out["train"].extend(rows[n_val + n_test :])
    # `adapt` validates the validation split, and `evaluate` each evaluated split, against the same
    # contract as the whole dataset (>= MIN_RECORDS, >= MIN_RECORDS_PER_CLASS per class). Refuse here,
    # before any model runs, rather than failing late with a message about "the dataset".
    for name in ("validation", "test"):
        part = out[name]
        per_class = {c: sum(1 for r in part if r["label"] == c) for c in manifest["classes"]}
        if len(part) < MIN_RECORDS or min(per_class.values()) < MIN_RECORDS_PER_CLASS:
            need = minimum_records_per_class(len(manifest["classes"]), val_fraction, test_fraction)
            raise ValueError(
                f"the {name} split would have {len(part)} records ({per_class}); each evaluated split needs "
                f"at least {MIN_RECORDS} records and {MIN_RECORDS_PER_CLASS} per class. With "
                f"{len(manifest['classes'])} balanced classes at val_fraction={val_fraction} and "
                f"test_fraction={test_fraction}, supply at least {need} records per class "
                f"({need * len(manifest['classes'])} in total)"
            )
    for part in out.values():
        rng.shuffle(part)
    return out


def minimum_records_per_class(n_classes: int, val_fraction: float = 0.2, test_fraction: float = 0.25) -> int:
    """Smallest per-class count for which a balanced dataset passes `split_dataset` and every downstream
    split check: each class contributes >= MIN_RECORDS_PER_CLASS to validation and test, each of those
    splits reaches MIN_RECORDS in total, and at least one record per class is left for training.
    For two classes at the default fractions this is 18 (36 records)."""
    for n in range(1, MAX_RECORDS + 1):
        n_val = max(1, round(n * val_fraction))
        n_test = max(1, round(n * test_fraction))
        if (
            n - n_val - n_test >= 1
            and min(n_val, n_test) >= MIN_RECORDS_PER_CLASS
            and min(n_val, n_test) * n_classes >= MIN_RECORDS
        ):
            return n
    raise ValueError("no dataset size satisfies the split contract at these fractions")


def load_byod_dataset(source: str | Path) -> list[dict[str, Any]]:
    """Read a user-supplied dataset (CSV with `id,smiles,label`, JSON array, or JSONL).

    SMILES are stripped of surrounding whitespace only; nothing else is rewritten (VAL7). The
    records are then validated with `validate_dataset`, whose errors name the offending row.
    """
    path = Path(source)
    if not path.is_file():
        raise FileNotFoundError(f"BYOD dataset file not found: {path}")
    text = path.read_text(encoding="utf-8-sig")
    if not text.strip():
        raise ValueError(f"BYOD dataset file is empty: {path}")
    suffix = path.suffix.lower()
    records: list[dict[str, Any]] = []
    if suffix == ".csv":
        reader = csv.DictReader(text.splitlines())
        header = [h.strip() for h in (reader.fieldnames or [])]
        missing = [c for c in REQUIRED_COLUMNS if c not in header]
        if missing:
            raise ValueError(f"CSV header {header} is missing required column(s) {missing}")
        for row in reader:
            records.append({c: (row.get(c) or "").strip() for c in REQUIRED_COLUMNS})
    elif suffix == ".jsonl":
        for line_no, line in enumerate(text.splitlines(), start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"line {line_no} is not valid JSON: {exc}") from exc
    elif suffix == ".json":
        try:
            data = json.loads(text)
        except json.JSONDecodeError as exc:
            raise ValueError(f"file is not valid JSON: {exc}") from exc
        if not isinstance(data, list):
            raise TypeError("JSON dataset must be a top-level array of objects")
        records = data
    else:
        raise ValueError(f"unsupported BYOD file type {suffix!r}; use .csv, .json or .jsonl")
    for rec in records:
        if isinstance(rec, Mapping) and isinstance(rec.get("smiles"), str):
            rec["smiles"] = rec["smiles"].strip()
    validate_dataset(records)
    return [dict(r) for r in records]


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write records as the BYOD CSV shape (`id,smiles,label`), so users have a template."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(REQUIRED_COLUMNS)
        for r in records:
            writer.writerow([r["id"], r["smiles"], r["label"]])
    return out

**Module 3/3:** `src/molformer_chemistry_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Classification metrics and trivial baselines for MoLFormer property-classification adaptation.

Pure Python (no scikit-learn): accuracy, macro-F1, per-class precision/recall/F1/support, and AUROC
(binary: positive class = the last entry of `classes`; multiclass: macro one-vs-rest), computed by
the Mann-Whitney rank statistic with average ranks for ties.
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .samples import formula_string` removed — names are kernel globals defined by the carried modules


def _prf(hits: int, n_pred: int, n_true: int) -> dict[str, float]:
    precision = hits / n_pred if n_pred else 0.0
    recall = hits / n_true if n_true else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)}


def auroc(y_true: Sequence[int], scores: Sequence[float]) -> float | None:
    """Area under the ROC curve for binary 0/1 labels; None when only one class is present."""
    n_pos = sum(1 for y in y_true if y == 1)
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    order = sorted(range(len(scores)), key=lambda i: scores[i])
    ranks = [0.0] * len(scores)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and scores[order[j + 1]] == scores[order[i]]:
            j += 1
        avg = (i + j + 2) / 2.0  # 1-based average rank of the tie block
        for k in range(i, j + 1):
            ranks[order[k]] = avg
        i = j + 1
    rank_sum = sum(r for r, y in zip(ranks, y_true, strict=True) if y == 1)
    return round((rank_sum - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg), 4)


def classification_metrics(
    y_true: Sequence[str],
    y_pred: Sequence[str],
    scores: Sequence[Sequence[float]] | None,
    classes: Sequence[str],
) -> dict[str, Any]:
    """Discrete and ranking metrics over one evaluation split (labels are class names).

    `scores[i][k]` is the score of class `classes[k]` for record i (softmax outputs from the
    pipeline; any monotone score works for AUROC). Class order is preserved exactly as given.
    """
    if len(y_true) != len(y_pred):
        raise ValueError(f"{len(y_true)} labels vs {len(y_pred)} predictions")
    class_list = list(classes)
    unknown = sorted((set(y_true) | set(y_pred)) - set(class_list))
    if unknown:
        raise ValueError(f"labels outside the class list {class_list}: {unknown}")
    n = len(y_true)
    correct = sum(1 for t, p in zip(y_true, y_pred, strict=True) if t == p)
    per_class: dict[str, dict[str, Any]] = {}
    f1s: list[float] = []
    for c in class_list:
        hits = sum(1 for t, p in zip(y_true, y_pred, strict=True) if t == c and p == c)
        n_pred = sum(1 for p in y_pred if p == c)
        n_true = sum(1 for t in y_true if t == c)
        prf = _prf(hits, n_pred, n_true)
        per_class[c] = {**prf, "support": n_true, "predicted": n_pred}
        if n_true:
            f1s.append(prf["f1"])
    result: dict[str, Any] = {
        "n": n,
        "accuracy": round(correct / n, 4) if n else 0.0,
        "macro_f1": round(sum(f1s) / len(f1s), 4) if f1s else 0.0,
        "per_class": per_class,
        "classes": class_list,
        "decision_rule": "argmax over class scores",
        "auroc": None,
        "auroc_definition": None,
    }
    if scores is not None and n:
        if len(scores) != n or any(len(row) != len(class_list) for row in scores):
            raise ValueError("scores must be one row per record with one column per class")
        if len(class_list) == 2:
            pos = class_list[-1]
            result["auroc"] = auroc([1 if t == pos else 0 for t in y_true], [row[-1] for row in scores])
            result["auroc_definition"] = f"binary AUROC with positive class {pos!r} (last class in the list)"
        else:
            values = []
            for k, c in enumerate(class_list):
                a = auroc([1 if t == c else 0 for t in y_true], [row[k] for row in scores])
                if a is not None:
                    values.append(a)
            result["auroc"] = round(sum(values) / len(values), 4) if values else None
            result["auroc_definition"] = "macro-averaged one-vs-rest AUROC over classes present in the split"
    return result


def majority_baseline(
    train_records: Sequence[Mapping[str, Any]],
    eval_records: Sequence[Mapping[str, Any]],
    classes: Sequence[str],
) -> dict[str, Any]:
    """Predict the most frequent training class for every evaluation record (EVAL11)."""
    counts: dict[str, int] = {}
    for r in train_records:
        counts[r["label"]] = counts.get(r["label"], 0) + 1
    majority = max(sorted(counts), key=counts.__getitem__)
    metrics = classification_metrics(
        [r["label"] for r in eval_records], [majority] * len(eval_records), None, classes
    )
    return {"baseline": "majority-class", "predicted_label": majority, **metrics}


def formula_baseline(
    train_records: Sequence[Mapping[str, Any]],
    eval_records: Sequence[Mapping[str, Any]],
    classes: Sequence[str],
) -> dict[str, Any]:
    """Predict from the molecular formula alone, learned on the training split only (SPL8).

    Molecular formula is the confounder worth ruling out in any molecular-property task: if the
    formula already separates the classes, the model has not had to learn chemistry. The rule is a
    lookup -- for each formula seen in training, predict its majority class; formulas never seen in
    training fall back to the overall training majority. On the tutorial sample every formula appears
    in both classes by construction (the pairs are constitutional isomers), so this baseline is
    expected to sit at chance.
    """
    class_list = list(classes)
    by_formula: dict[str, dict[str, int]] = {}
    overall: dict[str, int] = {}
    for record in train_records:
        formula = formula_string(record["smiles"])
        by_formula.setdefault(formula, {})
        by_formula[formula][record["label"]] = by_formula[formula].get(record["label"], 0) + 1
        overall[record["label"]] = overall.get(record["label"], 0) + 1
    fallback = max(sorted(overall), key=overall.__getitem__)
    lookup = {formula: max(sorted(counts), key=counts.__getitem__) for formula, counts in by_formula.items()}
    predicted, unseen = [], 0
    for record in eval_records:
        formula = formula_string(record["smiles"])
        if formula in lookup:
            predicted.append(lookup[formula])
        else:
            predicted.append(fallback)
            unseen += 1
    metrics = classification_metrics([r["label"] for r in eval_records], predicted, None, class_list)
    return {
        "baseline": "molecular-formula lookup",
        "distinct_train_formulas": len(lookup),
        "ambiguous_train_formulas": sum(1 for counts in by_formula.values() if len(counts) > 1),
        "eval_formulas_unseen_in_train": unseen,
        "fallback_label": fallback,
        **metrics,
    }

## 3. Pin, stage and verify the model

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `7`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `361063d0ad52…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `MolformerPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. **This load executes remote code.** `MolformerPipeline.from_pretrained` loads the tokenizer as a native fast tokenizer (`PreTrainedTokenizerFast`, which Transformers 5 prints as `TokenizersBackend`; `trust_remote_code=False`), but the model with `trust_remote_code=True`: Transformers imports and runs the checkpoint's own `configuration_molformer.py` and `modeling_molformer.py`, two of the 7 files verified above. That is unavoidable for this checkpoint: its `config.json` declares `model_type: "molformer"`, which Transformers has no native implementation of, so `trust_remote_code=False` refuses to load it. The SHA-256 check proves the code that runs is byte-for-byte the code at the pinned revision; it does **not** prove that code is safe or correct. Nobody has security-reviewed it here, so treat it like any third-party package you install. There is no fallback to a different download.

**Why `deterministic_eval` is overridden.** MoLFormer's linear attention uses random feature maps. The upstream `config.json` ships `deterministic_eval: false`, which redraws those features on every forward pass, so the same molecule gets a slightly different vector (about 1e-3 apart) on each call. This package loads with `deterministic_eval=True`, which keeps the checkpoint's feature buffers fixed in evaluation, so repeated inference is reproducible; Section 6 checks it. Training (Section 8) still redraws them, as upstream intends.

Depending on its logging settings, Transformers may also print two expected messages during this load: a note about instantiating a `molformer` model through `AutoModel`, and the checkpoint's `lm_head.*` weights reported as unused (they belong to the masked-language-model head, which embeddings and classification do not need). The cell ends with a one-line load summary. The effective identity, device and weight source are printed before any inference.

In [ ]:
# @title Infrastructure: stage and digest-verify the pinned snapshot, then load the model
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "molformer-xl-both-10pct",
  "modelId": "ibm-research/MoLFormer-XL-both-10pct",
  "revision": "361063d0ad524ef77cf39b08469f6be770dc550f",
  "files": [
    {
      "path": "README.md",
      "bytes": 6279,
      "sha256": "fff331c6a973fa6995662ac31354c702ac0f4805d494e7506b979a3a6cee6f44"
    },
    {
      "path": "config.json",
      "bytes": 1015,
      "sha256": "3ef9eaac8c7ca6282fd6256ed038d151bd4ff42a4ff855367e0d7197bbc1c284"
    },
    {
      "path": "configuration_molformer.py",
      "bytes": 7101,
      "sha256": "b88ea8d4b7b5e54f4f186cc7a230eff308928020a030f153a038fd12c05e3bed"
    },
    {
      "path": "model.safetensors",
      "bytes": 187248784,
      "sha256": "0795977fe7192c4acdaf052f0e8464af57bc4bb59211271c5e61aaba2637b9c6"
    },
    {
      "path": "modeling_molformer.py",
      "bytes": 36884,
      "sha256": "6f1ef72022de2c69e95661899422a7bb39a40a2cc5a6cb6216f14e9b7d84559c"
    },
    {
      "path": "tokenizer.json",
      "bytes": 54010,
      "sha256": "3df1f2219653c44fac9fa03b7f788b372eb2544ecc176737bb9aca8411b471a5"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 1132,
      "sha256": "56f93dc43e4383fcfc6342e09d782ef38d48ef227215a4a92ff8762fd5b8ff3e"
    }
  ],
  "totalBytes": 187355205
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = MolformerPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})
with open(WEIGHTS_DIR / 'config.json', encoding='utf-8') as handle:
    upstream_deterministic_eval = json.load(handle).get('deterministic_eval')
print({'remote_code_executed': True, 'remote_code_files_verified': list(REMOTE_CODE_FILES), 'tokenizer': type(pipe.tokenizer).__name__, 'upstream_deterministic_eval': upstream_deterministic_eval, 'deterministic_eval': DETERMINISTIC_EVAL, 'parameters': sum(p.numel() for p in pipe.model.parameters()), 'load_warnings': len(pipe.load_warnings)})

{'model_id': 'ibm-research/MoLFormer-XL-both-10pct', 'revision': '361063d0ad524ef77cf39b08469f6be770dc550f', 'license': 'apache-2.0', 'files': 7, 'total_bytes': 187355205}


{'weights_dir': '/content/weights/molformer-xl-both-10pct', 'fetched': ['README.md', 'config.json', 'configuration_molformer.py', 'model.safetensors', 'modeling_molformer.py', 'tokenizer.json', 'tokenizer_config.json']}


{'verified_files': 7, 'revision': '361063d0ad524ef77cf39b08469f6be770dc550f'}


[transformers] You are using a model of type `molformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights: 100%|██████████| 207/207 [00:00<00:00, 6020.16it/s]
[transformers] MolformerModel LOAD REPORT from: /content/weights/molformer-xl-both-10pct
Key                                | Status     |  | 
-----------------------------------+------------+--+-
lm_head.transform.LayerNorm.weight | UNEXPECTED |  | 
lm_head.transform.LayerNorm.bias   | UNEXPECTED |  | 
lm_head.transform.dense.weight     | UNEXPECTED |  | 
lm_head.decoder.weight             | UNEXPECTED |  | 
lm_head.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'device': 'cuda:0', 'source': 'local-snapshot'}
{'remote_code_executed': True, 'remote_code_files_verified': ['configuration_molformer.py', 'modeling_molformer.py'], 'tokenizer': 'TokenizersBackend', 'upstream_deterministic_eval': False, 'deterministic_eval': True, 'parameters': 44375040, 'load_warnings': 0}


## 4. Sample molecules, validation and split

The default dataset is generated in code with a fixed seed: 32 pairs, each a linear molecule and a branched **constitutional isomer** of it. Both members have the same molecular formula and the same terminal group; they differ only in where one carbon sits. `validate_dataset` checks the schema, the SMILES syntax, duplicate ids and SMILES, and class coverage before any model runs, and reports how many distinct formulas there are and how many of them appear in **both** classes — on the sample, all of them, which is the property the whole comparison rests on. `split_dataset` shuffles within each class and cuts 20 % validation / 25 % test, then refuses the split if the validation or test part would fall below the contract (8 records, 3 per class) that Sections 8 and 9 check, so a too-small BYOD file stops here, with the minimum it needs, not after training.

Look for: 64 molecules, classes `['branched', 'linear']`, 32 distinct formulas all shared across classes, splits 36/12/16, and a written `outputs/molformer_chemistry_sample_dataset.csv` — the file shape BYOD expects (with BYOD on, your validated records are written to `outputs/molformer_chemistry_byod_dataset.csv` instead). One honest caveat printed with them: a branched SMILES contains `(` and `)`, so a *character-level* baseline could separate these classes trivially; the baseline this notebook ships counts atoms, because molecular formula is the chemically meaningful confounder to rule out.

**Predict before running:** how many molecules, and how many distinct molecular formulas, will the sample have? If you knew only a molecule's formula, how well could you guess its class?

In [ ]:
import json
import os
from pathlib import Path

USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ''  # @param {type:"string"}
VAL_FRACTION = 0.2  # @param {type:"number"}
TEST_FRACTION = 0.25  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}

def byod_file(path, kind, suffixes=()):
    """BYOD path first (works on Colab, Kaggle and Jupyter); on Colab an empty path opens the upload dialog."""
    if str(path).strip():
        source = Path(str(path).strip()).expanduser()
        if not source.is_file():
            raise FileNotFoundError(f'BYOD path {str(source)!r} does not exist or is not a file (relative paths start at {os.getcwd()}); give the path of one {kind}.')
    else:
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError(f'BYOD is on but its path field is empty, and the upload dialog exists only in Google Colab: copy the {kind} into this runtime (or attach it as a Kaggle dataset) and set the path field.') from None
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError(f'Upload exactly one {kind} (received {len(uploaded)} files; a cancelled dialog sends none). Run this cell again.')
        name, payload = next(iter(uploaded.items()))
        source = Path('work') / Path(name).name
        source.parent.mkdir(parents=True, exist_ok=True)
        source.write_bytes(payload)
    if suffixes and not source.name.lower().endswith(tuple(suffixes)):
        raise ValueError(f'{source.name}: expected a {kind} ending in {" or ".join(suffixes)}.')
    return source


os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    byod_path = byod_file(BYOD_PATH, 'labelled SMILES file (.csv, .json or .jsonl)', ('.csv', '.json', '.jsonl'))
    records = load_byod_dataset(byod_path)
    data_source = 'BYOD (' + byod_path.name + ')'
else:
    records = generate_sample_dataset()
    data_source = f'synthetic constitutional-isomer dataset (seed {SAMPLE_SEED}, {SAMPLE_SIZE} molecules)'

dataset_manifest = validate_dataset(records)
CLASSES = dataset_manifest['classes']
splits = split_dataset(records, val_fraction=VAL_FRACTION, test_fraction=TEST_FRACTION, seed=SEED)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
dataset_csv = 'outputs/molformer_chemistry_byod_dataset.csv' if USE_BYOD else 'outputs/molformer_chemistry_sample_dataset.csv'
write_dataset_csv(records, dataset_csv)

print({'data_source': data_source, 'n_records': dataset_manifest['n_records'], 'classes': CLASSES, 'class_counts': dataset_manifest['class_counts']})
print({'distinct_formulas': dataset_manifest['distinct_formulas'], 'formulas_shared_across_classes': dataset_manifest['formulas_shared_across_classes'], 'validation': dataset_manifest['validation']})
print({'smiles_characters': dataset_manifest['smiles_characters'], 'ceilings': dataset_manifest['ceilings'], 'digest': dataset_manifest['digest'][:16] + '...'})
print({'train': len(train_records), 'validation': len(val_records), 'test': len(test_records), 'written': dataset_csv})

if not USE_BYOD:
    example_linear = next(r for r in records if r['label'] == 'linear')
    example_branched = next(r for r in records if r['id'].endswith(example_linear['id'][-3:]) and r['label'] == 'branched')
    print({'pair': (example_linear['smiles'], example_branched['smiles']), 'formula': (formula_string(example_linear['smiles']), formula_string(example_branched['smiles']))})
    assert formula_string(example_linear['smiles']) == formula_string(example_branched['smiles'])
    print('caveat: the branched SMILES contains ( and ), so a character-level baseline would separate these classes trivially; the formula baseline below deliberately counts atoms instead')

{'data_source': 'synthetic constitutional-isomer dataset (seed 20260918, 64 molecules)', 'n_records': 64, 'classes': ['branched', 'linear'], 'class_counts': {'branched': 32, 'linear': 32}}
{'distinct_formulas': 32, 'formulas_shared_across_classes': 32, 'validation': 'syntactic SMILES checks only; no cheminformatics toolkit is used'}
{'smiles_characters': {'min': 7, 'max': 18, 'mean': 13.0}, 'ceilings': {'max_tokens': 200, 'max_records': 5000, 'max_classes': 20, 'min_records': 8, 'min_records_per_class': 3}, 'digest': '7c09cc871b80568e...'}
{'train': 36, 'validation': 12, 'test': 16, 'written': 'outputs/molformer_chemistry_sample_dataset.csv'}
{'pair': ('CCCCCCCCCCCI', 'CC(C)CCCCCCCCI'), 'formula': ('C11I1', 'C11I1')}
caveat: the branched SMILES contains ( and ), so a character-level baseline would separate these classes trivially; the formula baseline below deliberately counts atoms instead


<details><summary>Check your reasoning</summary>64 molecules in two balanced classes split 36 / 12 / 16, with all 32 formulas shared across the classes. Because every linear molecule has a branched isomer with the same formula, knowing the formula alone tells you nothing about the class — that is the whole point of the construction, and Section 7 measures it.</details>

## 5. How a molecule becomes tokens

The tokenizer is a native `PreTrainedTokenizerFast` reading the checkpoint's own `tokenizer.json` — no remote code for this half. It splits SMILES into chemical tokens rather than characters: two-letter atoms such as `Cl` and `Br` stay one token, bracket atoms such as `[C@H]` stay one token, and branch parentheses and bond symbols are tokens of their own.

The ceiling that follows from the checkpoint is `MAX_TOKENS = 200` (its 202 position embeddings minus `<bos>` and `<eos>`). A molecule past that is **refused rather than truncated** — a truncated SMILES is a different molecule, not a shorter one — and the cell demonstrates that refusal along with four syntax rejections (an unclosed branch, a character outside the accepted set, an unpaired ring digit and leading whitespace). The printed `tokens_incl_bos_eos` counts `<bos>` and `<eos>`; the 200-token ceiling does not.

**Predict before running:** how many tokens will `ClCCBr` become — six characters, or fewer?

In [ ]:
for smiles in ['CCCCCCCO', 'CCCC(C)CCO', 'ClCCBr', 'C[C@H](N)C(=O)O', 'c1ccccc1O']:
    print({'smiles': smiles, 'tokens_incl_bos_eos': pipe.token_count(smiles), 'formula': formula_string(smiles)})

print({'max_tokens': MAX_TOKENS, 'max_position_embeddings': MAX_POSITION_EMBEDDINGS, 'max_molecules_per_call': MAX_MOLECULES_PER_CALL})

for probe in ['CCCC(CCO', 'CC*C', 'c1ccccO', ' CCO']:
    try:
        validate_inputs([probe])
        print({'probe': probe, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': probe, 'rejected': str(exc)[:100]})
try:
    pipe.embed(['C' * (MAX_TOKENS + 1)])
except ValueError as exc:
    print({'too_long': str(exc)[:130]})

input_manifest = validate_inputs([r['smiles'] for r in test_records[:4]], names=[r['id'] for r in test_records[:4]], token_counter=pipe.token_count)
print({'verdict': input_manifest['verdict'], 'n_molecules': input_manifest['n_molecules'], 'token_ceiling_checked': input_manifest['token_ceiling_checked'], 'requires_remote_code': input_manifest['requires_remote_code']})

{'smiles': 'CCCCCCCO', 'tokens_incl_bos_eos': 10, 'formula': 'C7O1'}
{'smiles': 'CCCC(C)CCO', 'tokens_incl_bos_eos': 12, 'formula': 'C7O1'}
{'smiles': 'ClCCBr', 'tokens_incl_bos_eos': 6, 'formula': 'Br1C2Cl1'}
{'smiles': 'C[C@H](N)C(=O)O', 'tokens_incl_bos_eos': 13, 'formula': 'C3N1O2'}
{'smiles': 'c1ccccc1O', 'tokens_incl_bos_eos': 11, 'formula': 'C6O1'}
{'max_tokens': 200, 'max_position_embeddings': 202, 'max_molecules_per_call': 64}
{'probe': 'CCCC(CCO', 'rejected': "molecules[0] ('CCCC(CCO') failed SMILES syntax checks: 1 unclosed parenthesis(s)"}
{'probe': 'CC*C', 'rejected': "molecules[0] ('CC*C') failed SMILES syntax checks: characters outside the accepted SMILES set: ['*']"}
{'probe': 'c1ccccO', 'rejected': "molecules[0] ('c1ccccO') failed SMILES syntax checks: unpaired ring-closure digit(s): ['1']"}
{'probe': ' CCO', 'rejected': 'molecules[0] has leading or trailing whitespace; strip it first'}
{'too_long': 'molecules[0] tokenizes to 201 tokens; the ceiling is 200 (max_positio

<details><summary>Check your reasoning</summary>In the recorded run `CCCCCCCO` became 10 tokens (eight atoms plus `<bos>` and `<eos>`), `ClCCBr` 6 tokens — `Cl` and `Br` stay single tokens — and `C[C@H](N)C(=O)O` 13 tokens, with the bracket atom `[C@H]` kept whole. A 201-token molecule is refused, not truncated: a cut SMILES is a different molecule.</details>

## 6. Molecule embeddings (representation, not prediction)

`pipe.embed` runs the verified encoder and returns one 768-dimensional vector per molecule: the mean of the last hidden state over non-padding tokens. Embeddings are representations — they carry no label and no metric of their own; a downstream labelled task is what gives them meaning (EVAL9). The cell embeds eight validation molecules, writes them with their ids to `outputs/molformer_chemistry_embeddings.csv` (OUT4), and prints the mean cosine similarity within and between classes as an inspection, not an evaluation.

Because this package loads with `deterministic_eval=True`, embedding the same molecules twice gives exactly the same vectors: the cell embeds the eight molecules a second time, prints whether every number is identical and asserts it. Under the upstream default (`deterministic_eval: false`, Section 3) the second call would differ by about 1e-3.

A different, smaller effect is shown next to it: the first molecule embedded **on its own** differs from the same molecule inside the padded 8-molecule batch by around 1e-6. That is float32 rounding from padding and batch shape, not random features; the cell prints it against a stated tolerance of `1e-4` (far below the ~1e-3 random-feature drift) for inspection, not as a pass/fail check.

In [ ]:
import csv
import math

embed_records = val_records[:8]
embedding_result = pipe.embed([r['smiles'] for r in embed_records], names=[r['id'] for r in embed_records])
vectors = embedding_result['embeddings']
print({'n_molecules': embedding_result['n_molecules'], 'dimension': embedding_result['dimension'], 'tokens': embedding_result['tokens'], 'pooling': embedding_result['pooling']})

second_call = pipe.embed([r['smiles'] for r in embed_records], names=[r['id'] for r in embed_records])['embeddings']
identical = second_call == vectors
print({'deterministic_eval': DETERMINISTIC_EVAL, 'identical_vectors_on_a_second_identical_call': identical})
assert identical, 'deterministic_eval=True must make a repeated identical embed call return identical vectors'
alone = pipe.embed([embed_records[0]['smiles']])['embeddings'][0]
single_vs_batch = max(abs(a - b) for a, b in zip(alone, vectors[0]))
print({'single_molecule_vs_padded_batch_max_abs_diff': single_vs_batch, 'tolerance': 1e-4, 'within_tolerance': single_vs_batch <= 1e-4, 'cause': 'float32 rounding from padding and batch shape, not random features'})

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

within, between = [], []
for i in range(len(embed_records)):
    for j in range(i + 1, len(embed_records)):
        sim = cosine(vectors[i], vectors[j])
        (within if embed_records[i]['label'] == embed_records[j]['label'] else between).append(sim)
print({'mean_cosine_within_class': round(sum(within) / len(within), 4) if within else None, 'mean_cosine_between_classes': round(sum(between) / len(between), 4) if between else None, 'note': 'inspection only; embeddings are unlabelled representations'})

with open('outputs/molformer_chemistry_embeddings.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'smiles', 'label'] + [f'dim_{k}' for k in range(embedding_result['dimension'])])
    for r, vec in zip(embed_records, vectors):
        writer.writerow([r['id'], r['smiles'], r['label']] + [f'{x:.6f}' for x in vec])
print('wrote outputs/molformer_chemistry_embeddings.csv')

{'n_molecules': 8, 'dimension': 768, 'tokens': [10, 17, 11, 15, 12, 9, 15, 9], 'pooling': 'mean of the last hidden state over non-padding tokens'}
{'deterministic_eval': True, 'identical_vectors_on_a_second_identical_call': True}
{'single_molecule_vs_padded_batch_max_abs_diff': 1.3113021850585938e-06, 'tolerance': 0.0001, 'within_tolerance': True, 'cause': 'float32 rounding from padding and batch shape, not random features'}
{'mean_cosine_within_class': 0.8288, 'mean_cosine_between_classes': 0.6905, 'note': 'inspection only; embeddings are unlabelled representations'}
wrote outputs/molformer_chemistry_embeddings.csv


## 7. Baselines on the test split

Two trivial predictors set the floor before any training (EVAL10/EVAL11). `majority_baseline` predicts the most frequent training class — 0.5 on a balanced split. `formula_baseline` looks up each test molecule's **molecular formula** among the training formulas and predicts that formula's majority training class, falling back to the overall training majority for formulas it has not seen (SPL8: the lookup is built on the training split only).

On this sample the formula baseline is structurally helpless, and the printout says why: every formula occurs exactly twice — once as a linear molecule, once as its branched isomer — so a formula seen in training is a coin flip, and a formula that was split across train and test is unseen and falls back. That is the control working as intended, not a bug. On your own data, read this baseline first: if the formula already predicts your label, the model does not have to learn any chemistry to score well.

On the default sample the cell also prints a third, deliberately crude rule that is **not** a chemistry baseline: "a `(` in the SMILES means branched". Section 4 warned that the branch parentheses make the classes separable at the character level; this rule measures it, so you can read Section 9's score against it.

**Predict before running:** write down the test accuracy you expect from each baseline, and from the one-character rule.

In [ ]:
baseline_majority = majority_baseline(train_records, test_records, CLASSES)
print({k: baseline_majority[k] for k in ('baseline', 'predicted_label', 'accuracy', 'macro_f1')})
baseline_formula = formula_baseline(train_records, test_records, CLASSES)
print({k: baseline_formula[k] for k in ('baseline', 'distinct_train_formulas', 'ambiguous_train_formulas', 'eval_formulas_unseen_in_train', 'accuracy', 'macro_f1')})
if not USE_BYOD:
    rule_correct = sum(('(' in r['smiles']) == (r['label'] == 'branched') for r in test_records)
    print({'rule': 'one-character rule (default sample only): a ( in the SMILES means branched', 'correct': rule_correct, 'of': len(test_records), 'accuracy': round(rule_correct / len(test_records), 4)})

{'baseline': 'majority-class', 'predicted_label': 'branched', 'accuracy': 0.5, 'macro_f1': 0.3333}
{'baseline': 'molecular-formula lookup', 'distinct_train_formulas': 25, 'ambiguous_train_formulas': 11, 'eval_formulas_unseen_in_train': 9, 'accuracy': 0.25, 'macro_f1': 0.2}
{'rule': 'one-character rule (default sample only): a ( in the SMILES means branched', 'correct': 16, 'of': 16, 'accuracy': 1.0}


<details><summary>Check your reasoning</summary>In the recorded runs the majority baseline scored accuracy 0.5 and macro-F1 0.3333 on the balanced 16-molecule test split; the formula baseline scored 0.25 / 0.2 — below the majority, because formulas seen in training are coin flips and 9 test formulas were unseen and fell back. Neither can beat chance on this dataset by construction. The one-character rule, by contrast, scores 16 of 16: on this sample the class is visible in a single character, which matters when you read Section 9.</details>

## 8. Bounded fine-tuning

`pipe.adapt` builds `MolformerForSequenceClassification` from the verified checkpoint (the head is newly initialised — the load report says so), freezes every parameter except the head and the last `TRAINABLE_LAYERS` encoder layers, and runs AdamW with the hyperparameters below (FT4/FT6): tutorial values chosen for a few seconds of CPU, not production settings. Validation metrics are computed after each epoch for **monitoring only**; the final epoch's weights are kept, so no selection happens on the validation split (EVAL14). Training loss going down is optimisation evidence, not task-quality evidence (FT7) — Section 9 is where quality is measured.

One MoLFormer-specific detail: during training the model is in `train()` mode, so its linear-attention random features **are** redrawn each step regardless of `deterministic_eval`. That is upstream's intended stochastic regularisation; it also means the feature buffers at the end of training differ from the checkpoint's, which is why Section 10 exports them with the adapter.

In [ ]:
import time

EPOCHS = 4  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}
TRAINABLE_LAYERS = 2  # @param {type:"integer"}

started = time.perf_counter()
adapt_result = pipe.adapt(
    train_records,
    val_records,
    classes=CLASSES,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    trainable_layers=TRAINABLE_LAYERS,
    seed=SEED,
)
adapt_seconds = round(time.perf_counter() - started, 2)
print({'method': adapt_result['method'], 'trainable_parameters': adapt_result['trainable_parameters'], 'total_parameters': adapt_result['total_parameters'], 'precision': adapt_result['precision'], 'device': pipe.device, 'seconds': adapt_seconds})
for step in adapt_result['history']:
    print(step)

Loading weights: 100%|██████████| 207/207 [00:00<00:00, 6942.44it/s]
[transformers] MolformerForSequenceClassification LOAD REPORT from: /content/weights/molformer-xl-both-10pct
Key                                | Status     | 
-----------------------------------+------------+-
lm_head.transform.LayerNorm.weight | UNEXPECTED | 
lm_head.transform.LayerNorm.bias   | UNEXPECTED | 
lm_head.transform.dense.weight     | UNEXPECTED | 
lm_head.decoder.weight             | UNEXPECTED | 
lm_head.transform.dense.bias       | UNEXPECTED | 
classifier.dense2.weight           | MISSING    | 
classifier.dense.weight            | MISSING    | 
classifier.dense2.bias             | MISSING    | 
classifier.dense.bias              | MISSING    | 
classifier.out_proj.bias           | MISSING    | 
classifier.out_proj.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly ini

{'method': 'gradient fine-tuning (AdamW) of the classification head and the last 2 encoder layer(s)', 'trainable_parameters': 8275970, 'total_parameters': 45557762, 'precision': 'float32', 'device': 'cuda:0', 'seconds': 4.64}
{'epoch': 1, 'train_loss': 0.482904, 'n_batches': 5, 'val_accuracy': 1.0, 'val_macro_f1': 1.0}
{'epoch': 2, 'train_loss': 0.182822, 'n_batches': 5, 'val_accuracy': 1.0, 'val_macro_f1': 1.0}
{'epoch': 3, 'train_loss': 0.024827, 'n_batches': 5, 'val_accuracy': 1.0, 'val_macro_f1': 1.0}
{'epoch': 4, 'train_loss': 0.007268, 'n_batches': 5, 'val_accuracy': 1.0, 'val_macro_f1': 1.0}


## 9. Held-out evaluation

`pipe.evaluate` classifies every molecule of a split and reports `accuracy`, `macro_f1` (the unweighted mean of per-class F1, which exposes a model that ignores a class), per-class precision/recall/F1 with support, and `auroc` (ranking quality of the positive-class score, independent of the argmax threshold). The **test split** was never used for training or monitoring, so its numbers are the independent evidence (SPL6/SPL7). These are tutorial metrics on a synthetic 16-molecule split (EVAL6): one holdout, no dispersion estimate. The report, with both baselines and the deltas against them, is written to `outputs/molformer_chemistry_evaluation_report.json`.

**Predict before running:** after four epochs, will the fine-tuned model beat both baselines on the test split? By how much — and would a perfect score make you trust it more or less?

In [ ]:
val_metrics = pipe.evaluate(val_records)
test_metrics = pipe.evaluate(test_records)
print({'split': 'validation', **{k: val_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
print({'split': 'test', **{k: test_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
for cls_name, row in test_metrics['per_class'].items():
    print({'class': cls_name, **row})

evaluation_report = {
    'task': 'molecular-property classification (bounded fine-tuning of MoLFormer-XL)',
    'evidence': 'tutorial sample-sanity metrics on one stratified holdout; not a chemistry benchmark',
    'estimation': 'single train/validation/test split, seed ' + str(SEED) + ', no dispersion estimate',
    'data_source': data_source,
    'dataset_digest': dataset_manifest['digest'],
    'classes': CLASSES,
    'splits': {'train': len(train_records), 'validation': len(val_records), 'test': len(test_records)},
    'baselines': {'majority': baseline_majority, 'formula': baseline_formula},
    'validation_metrics': val_metrics,
    'test_metrics': test_metrics,
    'delta_vs_majority': {k: round(test_metrics[k] - baseline_majority[k], 4) for k in ('accuracy', 'macro_f1')},
    'adaptation': {k: v for k, v in adapt_result.items() if k != 'trainable_parameter_names'},
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/molformer_chemistry_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report, f, indent=2)
print({'delta_vs_majority': evaluation_report['delta_vs_majority'], 'report': 'outputs/molformer_chemistry_evaluation_report.json'})

{'split': 'validation', 'n': 12, 'accuracy': 1.0, 'macro_f1': 1.0, 'auroc': 1.0}
{'split': 'test', 'n': 16, 'accuracy': 1.0, 'macro_f1': 1.0, 'auroc': 1.0}
{'class': 'branched', 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'support': 8, 'predicted': 8}
{'class': 'linear', 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'support': 8, 'predicted': 8}
{'delta_vs_majority': {'accuracy': 0.5, 'macro_f1': 0.6667}, 'report': 'outputs/molformer_chemistry_evaluation_report.json'}


<details><summary>Check your reasoning</summary>In the recorded runs train loss fell from 0.4516 to 0.0027 over four epochs, validation accuracy was 1.0 from epoch 1, and the test split scored accuracy, macro-F1 and AUROC all 1.0 (n = 16) against 0.5 and 0.25 for the baselines. A perfect score on 16 synthetic molecules is a saturated tutorial task: it says the adaptation contract works, not that the model predicts any real property. And the one-character rule of Section 7 also scores 16 of 16, so this sample cannot tell "the model reads structure" from "the model reads the `(` token". Lowering `EPOCHS` to 1 or training the head alone still gives 1.0 here; the first optional experiment at the end uses a change that does move the score.</details>

## 10. Inference on new molecules, artifact export and fresh reload

`pipe.classify` returns, per molecule, the argmax `label`, its `score` and the full `scores` dictionary in class order. The scores are softmax outputs of a head trained on a few dozen molecules — **not calibrated probabilities** (UNC2); the only decision rule is argmax (UNC3).

`pipe.save_artifact` then writes the trained tensors **plus the 12 linear-attention feature buffers** as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the class order, the tensor names, which of them are serving state, the file size and SHA-256, and the adaptation configuration (OUT8/ART3). Those buffers are not an afterthought: training redraws them, inference depends on them, and an adapter without them does not reproduce the adapted model. `MolformerPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digests **before** deserialising, rebuilds the classifier and overlays the tensors — a fresh object from files, not the in-memory model (VER2). The cell asserts identical labels and scores within `1e-5` (VER4).

**Predict before running:** the reloaded pipeline is rebuilt from files on disk. Will its scores match the in-memory model exactly, approximately, or not at all?

In [ ]:
if USE_BYOD:
    new_records = test_records[:6]
    new_source = 'held-out BYOD test molecules (the first six of the test split, already scored in Section 9; not new data)'
else:
    new_records = generate_sample_dataset(seed=7, size=6)
    new_source = 'freshly generated isomer pairs (seed 7)'
inference_result = pipe.classify([r['smiles'] for r in new_records], names=[r['id'] for r in new_records])
predictions = inference_result['predictions']
print({'new_source': new_source, 'decision_rule': inference_result['decision_rule']})
n_match = 0
for p, r in zip(predictions, new_records):
    n_match += p['label'] == r['label']
    print({'id': p['id'], 'smiles': p['smiles'], 'predicted': p['label'], 'score': round(p['score'], 4), 'true_label': r['label']})
print({'matches': n_match, 'of': len(new_records), 'note': 'sanity check on generated labels, not an evaluation'})

with open('outputs/molformer_chemistry_predictions.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'smiles', 'tokens', 'predicted_label', 'score'] + [f'score_{c}' for c in CLASSES])
    for p in predictions:
        writer.writerow([p['id'], p['smiles'], p['tokens'], p['label'], f"{p['score']:.6f}"] + [f"{p['scores'][c]:.6f}" for c in CLASSES])

artifact_dir = Path('outputs/molformer_chemistry_adapter')
pipe.save_artifact(artifact_dir, metadata={'data_source': data_source, 'dataset_digest': dataset_manifest['digest'], 'test_metrics': {k: test_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
with open(artifact_dir / ARTIFACT_MANIFEST_NAME, encoding='utf-8') as f:
    artifact_manifest = json.load(f)
print({'format': artifact_manifest['format'], 'base_model': artifact_manifest['base_model'], 'requires_remote_code': artifact_manifest['requires_remote_code'], 'deterministic_eval': artifact_manifest['deterministic_eval'], 'n_tensors': len(artifact_manifest['tensors']), 'serving_state_tensors': len(artifact_manifest['serving_state_tensors']), 'files': artifact_manifest['files']})

reloaded_pipe = MolformerPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR)
reloaded_result = reloaded_pipe.classify([r['smiles'] for r in new_records], names=[r['id'] for r in new_records])
max_score_diff = 0.0
for before, after in zip(predictions, reloaded_result['predictions']):
    assert before['id'] == after['id'] and before['label'] == after['label'], f'reload parity failure on {before["id"]}'
    max_score_diff = max(max_score_diff, abs(before['score'] - after['score']))
assert max_score_diff < 1e-5, f'reload score drift {max_score_diff}'
print({'reload_parity': 'PASS', 'labels_equal': True, 'max_abs_score_diff': max_score_diff})

{'new_source': 'freshly generated isomer pairs (seed 7)', 'decision_rule': 'argmax over softmax(logits); scores are softmax outputs, not calibrated probabilities'}
{'id': 'lin-000', 'smiles': 'CCCCCCCN', 'predicted': 'linear', 'score': 0.9984, 'true_label': 'linear'}
{'id': 'bra-000', 'smiles': 'CCC(C)CCCN', 'predicted': 'branched', 'score': 1.0, 'true_label': 'branched'}
{'id': 'lin-001', 'smiles': 'CCCCCCCCCCCCCN', 'predicted': 'linear', 'score': 0.9989, 'true_label': 'linear'}
{'id': 'bra-001', 'smiles': 'CCCCCC(C)CCCCCCN', 'predicted': 'branched', 'score': 0.9997, 'true_label': 'branched'}
{'id': 'lin-002', 'smiles': 'CCCCCCCCCN', 'predicted': 'linear', 'score': 0.9981, 'true_label': 'linear'}
{'id': 'bra-002', 'smiles': 'CCC(C)CCCCCN', 'predicted': 'branched', 'score': 0.9999, 'true_label': 'branched'}
{'matches': 6, 'of': 6, 'note': 'sanity check on generated labels, not an evaluation'}
{'format': 'org.valcorza.molformer-chemistry.adapter.v1', 'base_model': {'model_id': 'ibm-rese

[transformers] You are using a model of type `molformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Loading weights: 100%|██████████| 207/207 [00:00<00:00, 5092.68it/s]
[transformers] MolformerModel LOAD REPORT from: /content/weights/molformer-xl-both-10pct
Key                                | Status     |  | 
-----------------------------------+------------+--+-
lm_head.transform.LayerNorm.weight | UNEXPECTED |  | 
lm_head.transform.LayerNorm.bias   | UNEXPECTED |  | 
lm_head.transform.dense.weight     | UNEXPECTED |  | 
lm_head.decoder.weight             | UNEXPECTED |  | 
lm_head.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architect

{'reload_parity': 'PASS', 'labels_equal': True, 'max_abs_score_diff': 0.0}


Loading weights: 100%|██████████| 207/207 [00:00<00:00, 5065.79it/s]
[transformers] MolformerForSequenceClassification LOAD REPORT from: /content/weights/molformer-xl-both-10pct
Key                                | Status     | 
-----------------------------------+------------+-
lm_head.transform.LayerNorm.weight | UNEXPECTED | 
lm_head.transform.LayerNorm.bias   | UNEXPECTED | 
lm_head.transform.dense.weight     | UNEXPECTED | 
lm_head.decoder.weight             | UNEXPECTED | 
lm_head.transform.dense.bias       | UNEXPECTED | 
classifier.dense2.weight           | MISSING    | 
classifier.dense.weight            | MISSING    | 
classifier.dense2.bias             | MISSING    | 
classifier.dense.bias              | MISSING    | 
classifier.out_proj.bias           | MISSING    | 
classifier.out_proj.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly ini

<details><summary>Check your reasoning</summary>Identical labels, and in the recorded runs a maximum absolute score difference of 0.0 for the 6 freshly generated molecules (all 6 classified correctly). Before the 12 linear-attention feature buffers were exported with the adapter, the same reload differed by 0.0023 — the buffers are serving state, not an afterthought.</details>

## 11. Result export and provenance

The last output, `outputs/molformer_chemistry_result.json`, gathers what a reader needs to interpret the files above: the notebook source revision, the model id, immutable revision and licence, **the fact that remote code was executed and which files were verified first**, the `deterministic_eval` override, the dataset source and digest, the adaptation configuration, baseline and held-out metrics, the new-molecule predictions, the artifact manifest, the reload-parity result, and the runtime versions and device (OUT6/OUT7). No credential is involved anywhere in this notebook, so none can leak into it (OUT10).

In [ ]:
import platform

result_payload = {
    'task': 'molecular-property classification adaptation (MoLFormer-XL)',
    'pipeline_class': 'MolformerPipeline',
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'remote_code_executed': True,
    'remote_code_files': list(REMOTE_CODE_FILES),
    'deterministic_eval': DETERMINISTIC_EVAL,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'notebook_source': NOTEBOOK_SOURCE,
    'data_source': data_source,
    'dataset_manifest': dataset_manifest,
    'embedding_summary': {'n_molecules': embedding_result['n_molecules'], 'dimension': embedding_result['dimension'], 'pooling': embedding_result['pooling']},
    'evaluation_report': evaluation_report,
    'inference': {'new_source': new_source, 'decision_rule': inference_result['decision_rule'], 'predictions': predictions},
    'artifact_format': ARTIFACT_FORMAT,
    'artifact_format_version': ARTIFACT_FORMAT_VERSION,
    'artifact_manifest': artifact_manifest,
    'reload_parity': {'labels_equal': True, 'max_abs_score_diff': max_score_diff},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'safetensors': safetensors.__version__,
        'device': pipe.device,
        'precision': 'float32',
    },
}
with open('outputs/molformer_chemistry_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

print('outputs/:')
for path in sorted(Path('outputs').rglob('*')):
    if path.is_file():
        print(f'  - {path.as_posix()} ({path.stat().st_size / 1024:.1f} KB)')

outputs/:
  - outputs/molformer_chemistry_adapter/adapter.safetensors (32429.8 KB)
  - outputs/molformer_chemistry_adapter/manifest.json (5.7 KB)
  - outputs/molformer_chemistry_embeddings.csv (63.2 KB)
  - outputs/molformer_chemistry_evaluation_report.json (4.4 KB)
  - outputs/molformer_chemistry_predictions.csv (0.4 KB)
  - outputs/molformer_chemistry_result.json (15.0 KB)
  - outputs/molformer_chemistry_sample_dataset.csv (2.0 KB)


## Interpretation and limits

The fine-tuned head separates molecules that share a molecular formula and differ only in where one carbon sits — which the formula baseline cannot do, by construction. That rules out composition as the explanation. It does **not** show that the model reads chemical structure in any deeper sense: on this sample a one-character rule (a `(` in the SMILES means branched) scores the same 16 of 16, so the result is equally consistent with the model reading the branch token. Use BYOD with labels that no single character gives away for a real test. The test split has 16 synthetic molecules, the metrics come from one seeded holdout with no dispersion estimate, and the classes are a generator rule rather than a measured property. So a perfect score says the adaptation contract works, not that MoLFormer predicts solubility, toxicity, binding affinity or any real endpoint.

Three things to carry to real data. **Splits:** analogues from one chemical series share a scaffold, so a random split lets the model memorise it — use a scaffold or cluster split and expect lower, truer numbers. **Validation:** this repository checks SMILES *syntax*, not chemistry; run RDKit over your set before you trust it, because a syntactically fine string can be a chemically impossible molecule and the pipeline will happily embed it. **Applicability domain:** the pretraining corpus is drug-like PubChem and ZINC chemistry, so inorganics, organometallics, polymers and very large molecules are out of distribution with no error to warn you — and the 200-token ceiling refuses them rather than truncating.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model **and the Python it executes**, validate the demonstrated dataset contract, execute bounded fine-tuning, evaluate against trivial baselines on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or chemical validity.

**Optional experiments (they do not affect the default path).** For each one: write down a prediction, change the one form field in Section 8, then re-run **Sections 8, 9, 10 and 11 in order** (all four, so the evaluation report, the adapter and `result.json` describe the same run), compare with the default run, and explain the difference. Restore the default value and re-run the same four sections to return.

1. **A step size too small to learn.** Set `LEARNING_RATE = 1e-6` (100 times smaller). *Predict:* will the test accuracy stay at 1.0? *What happened in our local CPU run:* train loss stayed near 0.8 for all four epochs (the default falls to about 0.003) and test accuracy dropped to 0.25, below the 0.5 majority baseline. With such small steps the newly initialised head hardly moves from its random start in 20 updates, so what it scores is that random start, not learning. Your exact numbers can differ on a GPU; the direction should not.
2. **Less training on a saturated task.** Set `EPOCHS = 1`, or `TRAINABLE_LAYERS = 0` to train the head alone. *Predict:* does the test score fall? It does not: both still score 1.0 on this sample (validation accuracy is already 1.0 after the first default epoch). On a task this easy the score cannot rank the settings; the training loss and the trainable-parameter count are what change.
3. **Your own data.** Bring a labelled set through BYOD and read the formula baseline first: if it already separates your classes, your labels may be predictable from composition alone.

`pipe.adapt` builds a fresh classifier from the verified checkpoint on every call, so re-running Section 8 with other values never continues the previous adaptation.

## Troubleshooting

- **Section 1 stops with "This notebook needs a Linux x86_64 runtime"** — use Google Colab, Kaggle or a Linux x86_64 Jupyter server.
- **The uv wheel fails its size/SHA-256 check, or a download in Section 1 times out** — run Section 1 again; a complete environment is reused and an incomplete one is finished. If it repeats, `files.pythonhosted.org` or `pypi.org` is blocked or altered.
- **You re-ran Section 1 on its own** — nothing is lost: it keeps the running worker and every variable. After a session restart, run from the top.
- **"The isolated environment's Python process exited"** — usually out of memory; restart the session and choose **Run all**.
- **Section 3 reports a size or SHA-256 mismatch (weights or one of the two `.py` files)** — the message names the file. Delete the folder Section 3 prints as `weights_dir` and run Section 3 again; never edit the remote-code files.
- **A SMILES is refused** — the message names the rule (character set, bracket balance, ring digits, or more than 200 tokens). Fix or drop that molecule; nothing is truncated.
- **BYOD: "BYOD path … does not exist"** — the path is relative to the working directory printed in the message.
- **BYOD: "the upload dialog exists only in Google Colab"** — on Kaggle or Jupyter, copy the file into the runtime and set `BYOD_PATH`.
- **BYOD: "Upload exactly one …"** — the dialog was cancelled or several files were chosen; run Section 4 again.
- **BYOD: a `validate_dataset` refusal** — it names the record and the rule (duplicate id or SMILES, fewer than 8 records or 3 per class, more than 20 classes).
- **BYOD: "the validation split would have … records"** — your file passed validation but is too small to split: the validation and test splits each need 8 records and 3 per class. The message gives the per-class count to aim for (18 per class, 36 in total, for two balanced classes at the default fractions).

## Glossary

- **SMILES** — a line notation for a molecule: atoms as letters, branches in parentheses, ring closures as digits.
- **Constitutional isomers** — molecules with the same formula and atoms but a different skeleton (here: one carbon moved to a methyl branch).
- **Token** — the unit the tokenizer produces: an atom (`Cl`, `[C@H]`), a bond or a parenthesis; `MAX_TOKENS = 200`.
- **Linear attention / random features** — MoLFormer's attention approximation; the random feature buffers are fixed for evaluation (`deterministic_eval`) and exported with the adapter.
- **Remote code (`trust_remote_code`)** — Python shipped with the checkpoint that the loader executes; here both files are digest-verified first.
- **Embedding** — a 768-dimensional vector per molecule (mean of the last hidden state); a representation, not a prediction.
- **Majority / formula baseline** — always the most frequent training class; the training-majority class of the molecule's formula.
- **Accuracy / macro-F1 / AUROC** — share correct; unweighted mean of per-class F1; ranking quality of the positive-class score, independent of the threshold.
- **Epoch / learning rate / trainable layers** — one pass over the training molecules; the AdamW step size; how many final encoder layers are trained with the head.
- **Validation vs test split** — monitored during training vs never touched until the final evaluation.
- **Saturated task** — one where the model scores at the ceiling, so the score no longer distinguishes better from worse settings.
- **Adapter / reload parity** — the trained tensors plus serving buffers, overlaid on the pinned base; the reloaded pipeline gives the same scores.
- **BYOD** — bring your own data: your labelled SMILES through the same cells.

## Conclusion (your notes)

Optional — fill in from **your** run:

- The baselines scored ___ (majority) and ___ (formula); the fine-tuned model scored ___ accuracy, ___ macro-F1, ___ AUROC on ___ test molecules.
- I would / would not trust this number for a real property because ___ (for example the split, the dataset size, or saturation).
- Reload parity: max score difference ___.
- One change I would make before using my own data: ___.

## References

- Repository README: https://github.com/kurtvalcorza/molformer-chemistry-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/molformer-chemistry-pipeline/blob/main/MODEL_CARD.md
- Upstream model: https://huggingface.co/ibm-research/MoLFormer-XL-both-10pct
- Upstream code: https://github.com/IBM/molformer
- Ross, J., Belgodere, B., Chenthamarakshan, V., Padhi, I., Mroueh, Y., Das, P. (2021). Large-Scale Chemical Language Representations Capture Molecular Structure and Properties. arXiv:2106.09553. https://arxiv.org/abs/2106.09553